# Классификатор по датчикам: сравнение способов и проверка по времени

Выберите списки способов, признаков и горизонтов в пункте 1, затем выполните ячейки
сверху вниз. Регрессоры не нужны. Балансировка по умолчанию отключена.

**Память:** признаки готовятся отдельно по котлам и сохраняются в кеш на диске.
Нет общей матрицы для всех типов котлов и всех горизонтов. В обучении одновременно
находится одна матрица одного эксперимента и одна модель. Меняется горизонт —
меняются метки, а не создаётся ещё одна копия признаков. После эксперимента память освобождается.

**Проверка:** следующие друг за другом тестовые периоды с фиксированными общими датами.
Validation расширяется назад до достаточного числа прошлых аварий; train остаётся раньше неё.
Для всех наборов признаков одного сравнения даты одинаковы. Test без аварий не исключается.
Если прошлых событий всё равно мало, видны состав выборок и явная причина.
Неудачный подбор F-порога не подменяется на 0.5; 0.5 остаётся фиксированным baseline.

**Основной вывод — модель в целом:** classification report и confusion matrix
по всей последовательной тестовой истории, общий график и непрерывный расчёт уведомлений.
Отдельные версии находятся в закрытой диагностике. AUTO выбирает набор признаков
по прошлой validation и проверяется на будущем test.

Четыре основных порога сохранены. В скрытых подпунктах — macro-F для обоих классов
и подборы по устойчивым эпизодам с ограничениями ложных тревог. Все оптимумы
выбираются только на validation. Итоговые test-метрики не используются для подбора.

Функции обучения включены в этот файл. Для экспорта на дашборд обновите также `training/lib`
из архива: там находятся `classifier_export.py` и переносимая реализация инференса.
Можно положить рядом прежний `05_Классификатор.ipynb`: из него автоматически
читается DATA_SETTINGS с вашими путями. Код и результаты из другого ноутбука не выполняются.


In [ ]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
from copy import deepcopy
import ast, json, os, sys, gc
import numpy as np
import pandas as pd
from IPython.display import display, HTML

TRAINING_DIR = next((p.resolve() for p in (Path.cwd(), Path.cwd()/"training")
                     if (p/"lib/monitoring_v3").is_dir()), None)
if TRAINING_DIR is None:
    raise FileNotFoundError("Откройте ноутбук из training или корня проекта.")
os.chdir(TRAINING_DIR)
for p in (TRAINING_DIR, TRAINING_DIR/"lib"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
from notebook_reports import notebook_config
from monitoring_v3.data import load_sources
print("Папка проекта:", TRAINING_DIR)


Следующие две свёрнутые ячейки содержат функции. Их нужно выполнить, редактировать не требуется.


In [ ]:
"""A separate, causal classifier on sensor windows; no detector outputs required.

This source is also embedded in 05_Классификатор_без_регрессора.ipynb so that
copying the notebook alone works with the previous monitoring library.
"""
from pathlib import Path
import hashlib
import importlib.metadata
import json
import warnings
import base64
from html import escape
from io import BytesIO
import numpy as np
import pandas as pd


DIRECT_DEFAULTS = dict(
    window_hours=24, lags_hours=[12, 24], trends_hours=[6, 24],
    delta_hours=[1, 6], switch_hours=[6, 24], include_since_start=True,
    include_startup=True, minimum_numeric_coverage=.7, horizon_hours=168,
    min_lead_hours=0, train_step_hours=6, seed=42, iterations=400,
    depth=4, learning_rate=.05, early_stopping_rounds=50, threads=2,
    threshold_metric='f2', minimum_precision=None, balance_train=False, event_weights=False,
    train_end=None, validation_end=None, test_end=None, restore=True,
    allow_single_class_evaluation=False, macro_thresholds=True,
    alarm_selection=False, alarm_persist_hours=3, alarm_merge_hours=12,
    alarm_budget=2., alarm_background_fraction=.05, alarm_candidates=61,
)


def direct_options(overrides=None):
    options={**DIRECT_DEFAULTS,**(overrides or {})}
    extra=set(options)-set(DIRECT_DEFAULTS)
    if extra:raise ValueError('Неизвестные параметры: '+str(sorted(extra)))
    if options['window_hours']!=24:raise ValueError('Этот контракт признаков рассчитан на окно 24 часа.')
    for name in ('lags_hours','trends_hours','delta_hours','switch_hours'):
        if any(h<=0 or h>24 for h in options[name]):raise ValueError(name+': допустимы часы от 0 до 24.')
    for name in ('train_step_hours','iterations','depth','early_stopping_rounds','threads','horizon_hours'):
        if options[name]<=0:raise ValueError(name+' должен быть положительным.')
    if options['min_lead_hours']<0 or options['min_lead_hours']>=options['horizon_hours']:
        raise ValueError('min_lead_hours должен быть меньше горизонта и неотрицательным.')
    if options['threshold_metric'] not in ('f05','f1','f2'):raise ValueError('threshold_metric: f05, f1 или f2.')
    if options['minimum_precision'] is not None and not 0<=options['minimum_precision']<=1:
        raise ValueError('minimum_precision: число от 0 до 1 или None.')
    if options['alarm_persist_hours']<=0 or options['alarm_merge_hours']<0 or options['alarm_budget']<0:
        raise ValueError('Неверные параметры подтверждения/бюджета тревог.')
    if not 0<=options['alarm_background_fraction']<=1 or options['alarm_candidates']<3:
        raise ValueError('Неверная доля фоновой тревоги / число кандидатов.')
    if not 0<options['minimum_numeric_coverage']<=1:raise ValueError('Неверная полнота измерений.')
    if bool(options['train_end'])!=bool(options['validation_end']):
        raise ValueError('Задайте обе границы train_end / validation_end или оставьте обе None.')
    return options


def _direct_category(series):
    def value(v):
        if pd.isna(v):return '__MISSING__'
        if isinstance(v,(float,np.floating)) and float(v).is_integer():return str(int(v))
        return str(v)
    return series.astype(object).map(value)


def _direct_slope(frame,hours,step):
    """OLS on actual elapsed hours; available observations only, not imputed zeros."""
    window=max(1,int(pd.Timedelta(hours=hours)/step))
    t=pd.Series((frame.index-frame.index[0])/pd.Timedelta('1h'),index=frame.index)
    count=frame.rolling(window,min_periods=1).count()
    tx=frame.mul(t,axis=0).rolling(window,min_periods=1).sum()
    sx=frame.rolling(window,min_periods=1).sum()
    st=frame.notna().astype(float).mul(t,axis=0).rolling(window,min_periods=1).sum()
    st2=frame.notna().astype(float).mul(t*t,axis=0).rolling(window,min_periods=1).sum()
    denominator=st2-st*st/count
    return ((tx-st*sx/count)/denominator.where(denominator>1e-8)).where(count>=2)


def direct_features(context,cfg,options,eligible_only=False,start=None):
    """Features for both fit and inference. No labels or future sensor values used."""
    options=direct_options(options)
    index=context['frame'].index
    if index.has_duplicates or not index.is_monotonic_increasing:
        raise ValueError('История должна иметь уникальный отсортированный индекс времени.')
    step=pd.Timedelta(cfg['step'])
    for h in [*options['lags_hours'],*options['delta_hours'],*options['switch_hours'],*options['trends_hours']]:
        if pd.Timedelta(hours=h)%step:raise ValueError('Окна и лаги должны делиться на шаг данных.')
    roles=context['roles']
    numeric=[c for c,r in roles.items() if r in ('target','condition','load')]
    categories=[c for c,r in roles.items() if r=='state']
    if any(str(c).strip().casefold() in ('состояние','_тип') for c in numeric+categories):
        raise ValueError('Служебная разметка ремонта не может быть входом классификатора.')
    if not numeric:raise ValueError(context['unit']+': нет числовых датчиков.')
    data=context['cleaned'].reindex(columns=numeric).apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)
    coverage=data.notna().mean(axis=1)
    modes=['работа','пуск'] if options['include_startup'] else ['работа']
    operation=context['regime']['mode'].isin(modes)
    evidence=context.get('regime_spec',{})
    auto_currents=any(a.get('source')=='early_past' and a.get('threshold') is not None
        and pd.notna(a.get('threshold')) for a in evidence.get('audit',[]))
    needs_reference=context.get('threshold_source')=='initial_past' or auto_currents
    reference_end=pd.Timestamp(context.get('preparation_reference_end',evidence.get('reference_end',index[0])))
    reference_ready=pd.Series(index>=reference_end if needs_reference else True,index=index)
    eligible=operation&coverage.ge(options['minimum_numeric_coverage'])&reference_ready
    contiguous=index.to_series().diff().eq(step)
    segment=(~eligible|~eligible.shift(fill_value=False)|~contiguous).cumsum()
    schema=[];blocks=[]
    def record(sensor,operation,kind='numeric'):
        prefix='num' if sensor in numeric else 'cat'
        name=f'{prefix}::{sensor}::{operation}'
        description={'value':'Текущее значение', 'min_24h':'Минимум доступных значений за 24 часа',
            'max_24h':'Максимум доступных значений за 24 часа', 'mean_24h':'Среднее за 24 часа',
            'std_24h':'Стандартное отклонение за 24 часа, ddof=0',
            'held_hours':'Наблюдаемое время без изменения; на начале участка нижняя оценка'}.get(operation)
        if description is None:
            hours=operation.rsplit('_',1)[-1]
            description=('Значение '+hours+' назад' if operation.startswith('lag_') else
                'Изменение за '+hours if operation.startswith('delta_') else
                'Наклон линейного тренда за '+hours+', единиц датчика в час' if operation.startswith('trend_') else
                'Количество известных переключений за '+hours)
        schema.append(dict(feature=name,sensor=sensor,operation=operation,description=description,kind=kind,
            units=context.get('declared_units',{}).get(sensor,'')))
        return name
    names={}
    for operation in ['value',*[f'lag_{h}h' for h in options['lags_hours']],
        'min_24h','max_24h','mean_24h','std_24h',
        *[f'trend_{h}h' for h in options['trends_hours']],
        *[f'delta_{h}h' for h in options['delta_hours']]]:
        names[operation]={c:record(c,operation) for c in numeric}
    cat_names={}
    for c in categories:
        cat_names[c]={'value':record(c,'value','category'),'held_hours':record(c,'held_hours')}
        for h in options['switch_hours']:cat_names[c][f'switches_{h}h']=record(c,f'switches_{h}h')
    # Iterate only eligible continuous runs, never thousands of individual stopped rows.
    for _,chunk in data.loc[eligible].groupby(segment.loc[eligible],sort=False):
        history_index=chunk.index
        if start is not None:
            if chunk.index[-1]<pd.Timestamp(start):continue
            chunk=chunk.loc[pd.Timestamp(start)-pd.Timedelta('24h'):]
        out=[];rolling=chunk.rolling(max(1,int(pd.Timedelta('24h')/step)),min_periods=1)
        out.append(chunk.rename(columns=names['value']).astype('float32'))
        for h in options['lags_hours']:
            out.append(chunk.shift(int(pd.Timedelta(hours=h)/step)).rename(columns=names[f'lag_{h}h']).astype('float32'))
        for statistic in ('min','max','mean'):
            out.append(getattr(rolling,statistic)().rename(columns=names[statistic+'_24h']).astype('float32'))
        std=rolling.std(ddof=0).where(rolling.count()>=2)
        out.append(std.rename(columns=names['std_24h']).astype('float32'))
        for h in options['trends_hours']:
            out.append(_direct_slope(chunk,h,step).rename(columns=names[f'trend_{h}h']).astype('float32'))
        for h in options['delta_hours']:
            out.append(chunk.diff(int(pd.Timedelta(hours=h)/step)).rename(columns=names[f'delta_{h}h']).astype('float32'))
        for c in categories:
            source=context['frame'][c].reindex(history_index)
            known=source.notna();values=_direct_category(source)
            previous=values.shift();previous_known=known.shift(fill_value=False)
            change=known&previous_known&values.ne(previous)
            consecutive=(~known|~previous_known|values.ne(previous)).cumsum()
            # Observed unchanged duration; at initial observation it is a lower bound.
            held=source.groupby(consecutive).cumcount()*step/pd.Timedelta('1h')
            fields={cat_names[c]['value']:values,cat_names[c]['held_hours']:held.where(known)}
            for h in options['switch_hours']:
                count=change.astype(float).rolling(max(1,int(pd.Timedelta(hours=h)/step)),min_periods=1).sum()
                fields[cat_names[c][f'switches_{h}h']]=count.where(known)
            out.append(pd.DataFrame(fields,index=history_index).reindex(chunk.index))
        blocks.append(pd.concat(out,axis=1))
    columns=[s['feature'] for s in schema]
    output_index=index[eligible] if eligible_only else index
    if start is not None:output_index=output_index[output_index>=pd.Timestamp(start)]
    features=pd.concat(blocks).reindex(output_index) if blocks else pd.DataFrame(index=output_index,columns=columns)
    features['unit']=str(context['unit'])
    schema.append(dict(feature='unit',sensor='',operation='boiler_id',description='Идентификатор котла',kind='category',units=''))
    if options['include_since_start']:
        features['since_start_hours']=context['regime']['since_start_hours'].where(eligible)
        schema.append(dict(feature='since_start_hours',sensor='',operation='time_since_start',description='Время после пуска; при неизвестном начале NaN',kind='numeric',units='h'))
    for entry in schema:
        c=entry['feature']
        if entry['kind']=='category':features[c]=_direct_category(features[c])
        else:features[c]=pd.to_numeric(features[c],errors='coerce').replace([np.inf,-np.inf],np.nan).astype('float32')
    meta=pd.DataFrame(dict(eligible=eligible,numeric_coverage=coverage,mode=context['regime']['mode'],
        preparation_ready=reference_ready),index=index).reindex(output_index)
    return features,meta,pd.DataFrame(schema)


def direct_dataset(contexts,cfg,options,progress=None):
    from monitoring_v3.interpretation import risk_labels
    options=direct_options(options);xs={};ms={};schemas=[];audits=[]
    for unit,context in contexts.items():
        if progress:progress(f'{unit}: подготовка признаков и исходов')
        x,meta,schema=direct_features(context,cfg,options)
        testing=context['frame'].get('_Тип',pd.Series('',index=meta.index)).isin(['ИСП','ЗРР'])
        index=meta.index[meta.eligible&~testing]
        observed=context['cleaned'][context['load']].notna()
        as_of=context['frame'].index[-1]
        y,g,audit,available=risk_labels(index,context['events'],options['horizon_hours'],as_of,as_of,
            observed=observed,min_lead_hours=options['min_lead_hours'],return_audit=True,return_available_at=True)
        meta['label']=y.reindex(meta.index)
        meta['label_group']=(unit+'::'+g.astype(str)).reindex(meta.index)
        meta['label_known_at']=available.reindex(meta.index)
        meta['source_end']=as_of
        audit.update(unit=unit,total_rows=len(meta),eligible_rows=int(meta.eligible.sum()),
            numeric_sensors=len([c for c,r in context['roles'].items() if r in ('load','condition','target')]),
            categorical_sensors=len([c for c,r in context['roles'].items() if r=='state']),
            features=len(x.columns),preparation_reference_end=context.get('preparation_reference_end'),
            excluded_reference_rows=int((~meta.preparation_ready).sum()),
            positive_rows=int(y.eq(1).sum()),negative_rows=int(y.eq(0).sum()),
            positive_events=int(g.loc[y.eq(1)].nunique()),unlabelled_rows=int(y.isna().sum()))
        xs[unit]=x;ms[unit]=meta;schemas.append(schema);audits.append(audit)
    def combine(parts):
        result=pd.concat(parts,names=['unit','time']).swaplevel().sort_index(kind='stable')
        result.index.names=['time','unit'];return result
    features=combine(xs);meta=combine(ms)
    schema=pd.concat(schemas,ignore_index=True).drop_duplicates('feature')
    for name in schema.loc[schema.kind.eq('category'),'feature']:
        features[name]=_direct_category(features[name])
    return features,meta,schema,pd.DataFrame(audits)


def _direct_masks(meta,train_end,validation_end,options):
    t=pd.DatetimeIndex(meta.index.get_level_values('time'))
    good=meta.eligible&meta.label.notna()
    gap=pd.Timedelta(hours=options['window_hours'])
    known=pd.to_datetime(meta.label_known_at)
    masks=dict(
        train=good&(t<train_end-gap)&known.le(train_end),
        validation=good&(t>=train_end)&(t<validation_end-gap)&known.le(validation_end),
        test=good&(t>=validation_end),
    )
    if options.get('test_end'):masks['test']&=t<pd.Timestamp(options['test_end'])
    # Even manual time boundaries cannot split one failure/background group.
    later=set(meta.loc[masks['test'],'label_group'])
    masks['validation']&=~meta.label_group.isin(later)
    later|=set(meta.loc[masks['validation'],'label_group'])
    masks['train']&=~meta.label_group.isin(later)
    return masks


def direct_split(meta,options):
    options=direct_options(options)
    t=pd.DatetimeIndex(meta.index.get_level_values('time'))
    positive=meta.loc[meta.eligible&meta.label.eq(1)]
    starts=pd.Series(positive.index.get_level_values('time'),index=positive.label_group).groupby(level=0).min().sort_values()
    if options['train_end']:
        a,b=pd.Timestamp(options['train_end']),pd.Timestamp(options['validation_end'])
        if a>=b:raise ValueError('train_end должен быть раньше validation_end.')
        masks=_direct_masks(meta,a,b,options)
        method='manual'
    else:
        candidates=pd.DatetimeIndex(sorted(starts.unique()))
        if len(candidates)>24:candidates=candidates[np.unique(np.linspace(0,len(candidates)-1,24).astype(int))]
        best=None
        for i,a in enumerate(candidates[1:-1],1):
            for b in candidates[i+1:]:
                masks0=_direct_masks(meta,a,b,options)
                counts=[];okay=True
                for name,mask in masks0.items():
                    if meta.loc[mask,'label'].nunique()!=2:okay=False;break
                    counts.append(meta.loc[mask&meta.label.eq(1),'label_group'].nunique())
                if not okay:continue
                fractions=np.array(counts)/max(len(starts),1)
                loss=float(np.square(fractions-[.6,.2,.2]).sum()+.05*(1-fractions.sum()))
                if best is None or loss<best[0]:best=(loss,a,b,masks0)
        if best is None:
            raise ValueError('Нельзя выделить независимые train/validation/test с обоими классами. '
                f'Доступно независимых аварий: {len(starts)}. Нужны аварии в трёх последовательных '
                'периодах с известными к границам исходами. Объедините котлы одной группы или задайте '
                'две границы вручную; случайное перемешивание часов не используется.')
        _,a,b,masks=best;method='chronological_event_counts'
    rows=[]
    for name,mask in masks.items():
        subset=meta.loc[mask];positive=subset.label.eq(1)
        rows.append(dict(partition=name,rows=len(subset),negative=int(subset.label.eq(0).sum()),
            positive=int(positive.sum()),positive_events=int(subset.loc[positive,'label_group'].nunique()),
            negative_groups=int(subset.loc[~positive,'label_group'].nunique()),
            first=t[mask].min() if mask.any() else pd.NaT,last=t[mask].max() if mask.any() else pd.NaT))
    summary=pd.DataFrame(rows)
    if meta.loc[masks['train'],'label'].nunique()!=2:
        raise ValueError('В обучении отсутствует один из классов:\n'+summary.to_string(index=False))
    if not masks['validation'].any():raise ValueError('Нет известных исходов в validation.')
    if not options['allow_single_class_evaluation'] and any(meta.loc[mask,'label'].nunique()!=2 for mask in masks.values()):
        raise ValueError('В одном из заданных периодов нет обоих классов:\n'+summary.to_string(index=False))
    return masks,summary,dict(train_end=str(a),validation_end=str(b),method=method,
        window_gap_hours=options['window_hours'],unassigned_labelled_rows=int(
            (meta.label.notna()&~(masks['train']|masks['validation']|masks['test'])).sum()))


def _direct_stratified_positions(positions,groups,amount,rng):
    """Round-robin groups, rather than sampling an entire quiet month."""
    if amount>=len(positions):return np.asarray(positions,dtype=int)
    buckets=[]
    for _,indices in pd.Series(positions,index=np.asarray(groups)).groupby(level=0,sort=False):
        values=indices.to_numpy().copy();rng.shuffle(values);buckets.append(list(values))
    rng.shuffle(buckets);selected=[]
    while len(selected)<amount:
        remaining=[]
        for bucket in buckets:
            if bucket:selected.append(bucket.pop())
            if bucket:remaining.append(bucket)
            if len(selected)==amount:break
        buckets=remaining
    return np.sort(selected)


def direct_balance(meta,train_mask,options):
    options=direct_options(options);t=pd.DatetimeIndex(meta.index.get_level_values('time'))
    period=pd.Timedelta(hours=options['train_step_hours']).value
    sample=(t.as_unit('ns').asi8%period)==0
    positions=np.flatnonzero(train_mask.to_numpy()&sample)
    counts=meta.iloc[positions].label.value_counts()
    if set(counts.index)!={0.,1.}:raise ValueError('После шага обучения нет обоих классов.')
    selected=positions
    if options['balance_train']:
        amount=int(counts.min());rng=np.random.default_rng(options['seed']);selected=[]
        for label in (0,1):
            pos=positions[meta.iloc[positions].label.eq(label).to_numpy()]
            selected.extend(_direct_stratified_positions(pos,meta.iloc[pos].label_group,amount,rng))
    selected=np.sort(selected);subset=meta.iloc[selected]
    weights=np.ones(len(selected),dtype=float)
    if options['balance_train'] and options['event_weights']:
        weights=1/subset.label_group.map(subset.label_group.value_counts()).to_numpy(float)
        totals=pd.Series(weights,index=subset.index).groupby(subset.label).sum()
        weights=weights/subset.label.map(totals).to_numpy(float)*len(weights)/2
    report=pd.DataFrame([dict(label=label,rows=int(subset.label.eq(label).sum()),
        independent_groups=int(subset.loc[subset.label.eq(label),'label_group'].nunique()),
        total_weight=float(weights[subset.label.eq(label).to_numpy()].sum())) for label in (0,1)])
    return selected,weights,report


def direct_threshold(y,scores,options):
    from sklearn.metrics import precision_recall_curve
    p,r,thresholds=precision_recall_curve(y,scores)
    table=pd.DataFrame(dict(threshold=thresholds,precision=p[:-1],recall=r[:-1]))
    for beta,name in ((.5,'f05'),(1,'f1'),(2,'f2')):
        denominator=beta*beta*table.precision+table.recall
        table[name]=(1+beta*beta)*table.precision*table.recall/denominator.replace(0,np.nan)
        table[name]=table[name].fillna(0.)
    table['allowed']=True
    table['selection_valid']=pd.Series(y).nunique()==2
    npos=int(np.asarray(y).sum());nneg=len(y)-npos
    table['tp']=np.rint(table.recall*npos).astype(int)
    # Explicit >= tie handling for leaf-constant CatBoost predictions.
    predicted=len(scores)-np.searchsorted(np.sort(np.asarray(scores)),table.threshold.to_numpy(),side='left')
    table['fp']=predicted-table.tp;table['fn']=npos-table.tp;table['tn']=nneg-table.fp
    if options['minimum_precision'] is not None:
        table['allowed']=table.precision.ge(options['minimum_precision'])
    allowed=table.loc[table.allowed&table.selection_valid]
    if allowed.empty:
        reason='validation_single_class' if pd.Series(y).nunique()!=2 else 'validation_precision_constraint_unmet'
        return np.inf,table,reason+': порог не подобран'
    chosen=allowed.sort_values([options['threshold_metric'],'precision','threshold'],ascending=False).iloc[0]
    return float(chosen.threshold),table,'selected_on_validation_'+options['threshold_metric']


def direct_thresholds(package):
    """Four decisions from the saved validation PR curve, including old artifacts."""
    table=package['threshold_table'].copy()
    for beta,name in ((.5,'f05'),(1,'f1'),(2,'f2')):
        denominator=beta*beta*table.precision+table.recall
        table[name]=((1+beta*beta)*table.precision*table.recall/denominator.replace(0,np.nan)).fillna(0.)
    if 'tp' not in table and 'split_summary' in package:
        val=package['split_summary'].loc[lambda d:d.partition.eq('validation')]
        if len(val):
            pos=int(val.positive.iloc[0]);neg=int(val.negative.iloc[0]);table['tp']=np.rint(table.recall*pos)
            table['fp']=(table.tp/table.precision.replace(0,np.nan)-table.tp).fillna(0).round()
            table['fn']=pos-table.tp;table['tn']=neg-table.fp
    if {'tp','tn','fp','fn'}.issubset(table):
        for beta,name in ((.5,'f05'),(1,'f1'),(2,'f2')):
            positive=(1+beta**2)*table.tp/((1+beta**2)*table.tp+beta**2*table.fn+table.fp).replace(0,np.nan)
            negative=(1+beta**2)*table.tn/((1+beta**2)*table.tn+beta**2*table.fp+table.fn).replace(0,np.nan)
            table['macro_'+name]=(positive.fillna(0)+negative.fillna(0))/2
    allowed=table.loc[table['allowed'].eq(True)] if 'allowed' in table else table
    if 'selection_valid' in allowed:allowed=allowed.loc[allowed.selection_valid]
    rules={'default_05':dict(threshold=.5,title='Стандартный порог 0.5',status='fixed')}
    for metric,title in [('f1','Максимум F1'),('f2','Максимум F2 · приоритет Recall'),
                         ('f05','Максимум F0.5 · приоритет Precision')]:
        row=allowed.sort_values([metric,'precision','threshold'],ascending=False).iloc[0] if len(allowed) else None
        rules['best_'+metric]=dict(threshold=float(row.threshold) if row is not None else np.inf,
            title=title,category='positive_class',available=row is not None,
            status='validation_'+metric if row is not None else 'threshold_unavailable: недостаточно классов или не выполнено ограничение Precision')
    if package.get('options',{}).get('macro_thresholds',False):
        for metric,title in [('macro_f1','Максимум macro F1 · оба класса'),('macro_f2','Максимум macro F2 · оба класса'),('macro_f05','Максимум macro F0.5 · оба класса')]:
            row=allowed.sort_values([metric,'precision','threshold'],ascending=False).iloc[0] if len(allowed) and metric in allowed else None
            rules['best_'+metric]=dict(threshold=float(row.threshold) if row is not None else np.inf,title=title,
                category='both_classes',available=row is not None,
                status='validation_'+metric if row is not None else 'threshold_unavailable: недостаточно классов в validation')
    rules.update(package.get('alarm_thresholds',{}))
    return rules,table


def direct_metrics(y,scores,threshold,predictions=None):
    from sklearn.metrics import (precision_score,recall_score,f1_score,fbeta_score,
        average_precision_score,roc_auc_score,log_loss,brier_score_loss,confusion_matrix,accuracy_score)
    y=np.asarray(y,dtype=int);scores=np.asarray(scores,dtype=float)
    if not len(y):return dict(rows=0,status='no_evaluable_rows',threshold=float(threshold))
    pred=scores>=threshold if predictions is None else np.asarray(predictions,dtype=bool)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return dict(rows=len(y),negative=int((y==0).sum()),positive=int(y.sum()),prevalence=float(y.mean()),threshold=float(threshold),
        precision=float(precision_score(y,pred,zero_division=0)) if pred.any() else None,
        recall=float(recall_score(y,pred,zero_division=0)) if y.any() else None,
        f05=float(fbeta_score(y,pred,beta=.5,zero_division=0)),
        f1=float(f1_score(y,pred,zero_division=0)),f2=float(fbeta_score(y,pred,beta=2,zero_division=0)),
        macro_f05=float(fbeta_score(y,pred,beta=.5,average='macro',labels=[0,1],zero_division=0)),
        macro_f1=float(fbeta_score(y,pred,beta=1,average='macro',labels=[0,1],zero_division=0)),
        macro_f2=float(fbeta_score(y,pred,beta=2,average='macro',labels=[0,1],zero_division=0)),
        specificity=float(tn/(tn+fp)) if tn+fp else None,
        average_precision=float(average_precision_score(y,scores)) if len(np.unique(y))==2 else None,
        roc_auc=float(roc_auc_score(y,scores)) if len(np.unique(y))==2 else None,
        random_roc_auc=.5,random_average_precision=float(y.mean()),
        logloss=float(log_loss(y,np.column_stack([1-scores,scores]),labels=[0,1])),
        brier=float(brier_score_loss(y,scores)),accuracy=float(accuracy_score(y,pred)),
        true_positive=int(tp),false_positive=int(fp),false_negative=int(fn),true_negative=int(tn),
        predicted_positive_fraction=float(pred.mean()))


def direct_native(features,columns,categorical):
    out={}
    for c in columns:
        values=features[c] if c in features else pd.Series(np.nan,index=features.index)
        out[c]=_direct_category(values) if c in categorical else pd.to_numeric(values,errors='coerce').replace([np.inf,-np.inf],np.nan).astype('float32')
    return pd.DataFrame(out,index=features.index)


def direct_alarm_evaluate(predictions,contexts,options,threshold,step='1h',as_of=None,start=None,end=None):
    """Causal confirmation and repeat suppression, including gaps/unknown outcomes."""
    from monitoring_v3.alarm_scores import alarm_stream
    from monitoring_v3.event_registry import event_type_at
    policy=dict(step=step,persist_hours=options['alarm_persist_hours'],merge_hours=options['alarm_merge_hours'])
    parts={};events=[];tp_groups=set();false=uncertain=positive_notifications=0;background=active_background=0
    for unit in predictions.index.get_level_values('unit').unique():
        p=predictions.xs(unit,level='unit').sort_index().copy()
        local_threshold=(threshold.xs(unit,level='unit').reindex(p.index)
            if isinstance(threshold,pd.Series) and isinstance(threshold.index,pd.MultiIndex) else threshold)
        threshold_known=np.isfinite(local_threshold)
        binary=pd.Series(np.where(p.score.notna()&threshold_known,np.where(p.score.ge(local_threshold),2.,0.),np.nan),index=p.index)
        stream,_=alarm_stream(binary,policy)
        p['alarm_active']=stream.active;p['alarm_notification']=stream.notification
        known=p.label.notna()&p.score.notna()&threshold_known
        if as_of is not None:known&=pd.to_datetime(p.label_known_at).le(pd.Timestamp(as_of))
        label=p.label.where(known)
        bg=label.eq(0);positive=label.eq(1)&p.alarm_notification
        false+=int((bg&p.alarm_notification).sum());uncertain+=int((~known&p.alarm_notification).sum())
        positive_notifications+=int(positive.sum());tp_groups.update(p.loc[positive,'label_group'].astype(str))
        background+=int(bg.sum());active_background+=int((bg&p.alarm_active).sum())
        if unit in contexts:
            c=contexts[unit];date=pd.Timestamp(as_of) if as_of is not None else c['frame'].index[-1]
            left=pd.Timestamp(start) if start is not None else p.index.min()
            right=pd.Timestamp(end) if end is not None else p.index.max()+pd.Timedelta(step)
            for e in c['events'].itertuples():
                kind,available=event_type_at(e,date)
                if kind!='Авария' or pd.isna(e.stop) or not left<=e.stop<right or pd.isna(available) or available>date:continue
                window=(p.index>=e.stop-pd.Timedelta(hours=options['horizon_hours']))&(p.index<e.stop)
                belongs=p.label_group.astype(str).str.endswith(str(e.event_id))
                notices=p.loc[window&belongs&positive]
                observed=p.loc[window&belongs&known]
                events.append(dict(unit=unit,event_id=e.event_id,stop=e.stop,detected=bool(len(notices)),
                    scored_points=len(observed),full_window_in_period=e.stop-pd.Timedelta(hours=options['horizon_hours'])>=left,
                    first_warning=notices.index[0] if len(notices) else pd.NaT,
                    lead_hours=float((e.stop-notices.index[0])/pd.Timedelta('1h')) if len(notices) else np.nan))
        parts[unit]=p
    combined=pd.concat(parts,names=['unit','time']).swaplevel().sort_index() if parts else predictions.copy()
    combined.index.names=['time','unit'];event_table=pd.DataFrame(events)
    hours=background*float(pd.Timedelta(step)/pd.Timedelta('1h'))
    recall=float(event_table.detected.mean()) if len(event_table) else None
    precision=len(tp_groups)/(len(tp_groups)+false) if len(tp_groups)+false else None
    metrics=dict(events=len(event_table),detected_events=int(event_table.detected.sum()) if len(event_table) else 0,
        event_recall=recall,notification_precision=precision,false_notifications=false,
        true_notifications=len(tp_groups),duplicate_notifications=positive_notifications-len(tp_groups),
        uncertain_notifications=uncertain,background_hours=hours,
        false_alarms_per_720h=false*720/hours if hours else None,
        background_alarm_fraction=active_background/background if background else None,
        unscored_events=int(event_table.scored_points.eq(0).sum()) if len(event_table) else 0,
        median_lead_hours=float(event_table.lead_hours.median()) if len(event_table) and event_table.lead_hours.notna().any() else None)
    metrics['event_f1']=2*precision*recall/(precision+recall) if precision is not None and recall is not None and precision+recall else 0.
    return metrics,combined,event_table


def direct_alarm_thresholds(predictions,contexts,options,step,as_of,start,end):
    scores=predictions.score.dropna().to_numpy()
    if not len(scores):return {},pd.DataFrame()
    candidates=np.unique(np.r_[np.quantile(scores,np.linspace(0,1,int(options['alarm_candidates']))),np.nextafter(scores.max(),np.inf)])
    rows=[]
    for threshold in candidates:
        metrics,_,_=direct_alarm_evaluate(predictions,contexts,options,float(threshold),step,as_of,start,end)
        rows.append(dict(threshold=float(threshold),**metrics))
    table=pd.DataFrame(rows)
    table['allowed']=(table.false_alarms_per_720h.le(options['alarm_budget'])&
        table.background_alarm_fraction.le(options['alarm_background_fraction']))
    rules={}
    for name,objective,title in [('alarm_budget_recall','event_recall','Эпизоды: максимум Recall при бюджете'),
        ('alarm_budget_f1','event_f1','Эпизоды: максимум F1 при бюджете')]:
        allowed=table.loc[table.allowed&table[objective].notna()&table.events.gt(0)]
        selected=allowed.sort_values([objective,'false_notifications','threshold'],ascending=[False,True,False]).iloc[0] if len(allowed) else None
        rules[name]=dict(threshold=float(selected.threshold) if selected is not None else np.inf,
            available=selected is not None,
            title=title,category='alarm_policy',status=('selected_on_validation_alarm_budget' if selected.detected_events>0 else 'constraints_met_but_no_events_detected') if selected is not None else 'threshold_unavailable: нет пригодного подбора по эпизодам',
            validation=dict(selected.to_dict()) if selected is not None else {})
    return rules,table


def direct_fit(features,meta,schema,options,folder,verbose=100,contexts=None,cfg=None,readiness=None):
    from catboost import CatBoostClassifier,Pool
    from monitoring_v3.threads import numerical_threads
    import joblib
    options=direct_options(options);folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
    masks,partitions,boundaries=direct_split(meta,options)
    if readiness:
        tr=partitions.loc[partitions.partition.eq('train')].iloc[0]
        va=partitions.loc[partitions.partition.eq('validation')].iloc[0]
        if (tr.positive_events<readiness['min_train_events'] or
            va.positive_events<readiness['min_validation_events'] or
            va.negative_groups<readiness['min_validation_negative_groups']):
            raise ValueError('После изоляции групп недостаточно независимых событий/фоновых групп. '+partitions.to_string(index=False))
    positions,weights,balance=direct_balance(meta,masks['train'],options)
    train=features.iloc[positions]
    varying=train.nunique(dropna=False).gt(1)&train.notna().any()
    columns=list(train.columns[varying]);dropped=list(train.columns[~varying])
    if not columns:raise ValueError('После подготовки нет изменяющихся входов.')
    categorical=[c for c in columns if c in set(schema.loc[schema.kind.eq('category'),'feature'])]
    frozen_preparation=None
    if contexts is not None and cfg is not None:
        from classifier_export import freeze_preparation
        frozen_preparation=freeze_preparation(contexts,cfg)
    signature=hashlib.sha256(json.dumps(dict(contract=5,options=options,boundaries=boundaries,
        preparation=frozen_preparation,
        alarm_step=(cfg or {}).get('step','1h'),
        event_registry={u:c['events'].to_json(date_format='iso') for u,c in (contexts or {}).items()} if options['alarm_selection'] else {},
        versions={p:importlib.metadata.version(p) for p in ('catboost','pandas','numpy','scikit-learn')}),sort_keys=True).encode())
    for frame in (features,meta):signature.update(pd.util.hash_pandas_object(frame,index=True).to_numpy().tobytes())
    token=signature.hexdigest();cache=folder/'classifier.joblib'
    if options['restore'] and cache.exists():
        try:cached=joblib.load(cache)
        except Exception:cached={}
        if cached.get('fingerprint')==token and all(k in cached for k in ('metrics','predictions','importance')):
            print('Модель и проверка восстановлены: данные и настройки совпадают.',flush=True)
            _direct_save_reports(cached,folder)
            return cached,masks,partitions,balance
    validation=features.loc[masks['validation']]
    model=CatBoostClassifier(iterations=int(options['iterations']),depth=int(options['depth']),
        learning_rate=options['learning_rate'],loss_function='Logloss',eval_metric=('PRAUC' if meta.loc[masks['validation'],'label'].nunique()==2 else 'Logloss'),
        random_seed=options['seed'],thread_count=int(options['threads']),has_time=True,
        allow_writing_files=False)
    train_pool=Pool(direct_native(train,columns,categorical),meta.iloc[positions].label.astype(int),
        cat_features=categorical,weight=weights)
    validation_pool=Pool(direct_native(validation,columns,categorical),
        meta.loc[masks['validation'],'label'].astype(int),cat_features=categorical)
    with numerical_threads(options['threads']):
        model.fit(train_pool,eval_set=validation_pool,use_best_model=True,
            early_stopping_rounds=int(options['early_stopping_rounds']),verbose=verbose)
    validation_scores=model.predict_proba(validation_pool,thread_count=options['threads'])[:,1]
    threshold,threshold_table,threshold_status=direct_threshold(
        meta.loc[masks['validation'],'label'].astype(int),validation_scores,options)
    # Persist weights and threshold before test plots/report rendering.
    model.save_model(str(folder/'classifier.cbm'))
    package=dict(model=model,columns=columns,categorical=categorical,options=options,
        feature_contract='telemetry_only_v1',
        schema=schema,boundaries=boundaries,threshold=threshold,threshold_status=threshold_status,
        dropped_columns=dropped,fingerprint=token,score_status=('uncalibrated_after_balanced_training'
            if options['balance_train'] else 'uncalibrated_natural_training'),
        available_at=boundaries['validation_end'],training_rows=len(positions),
        train_prevalence=float(meta.loc[masks['train'],'label'].mean()),
        split_summary=partitions,balance_summary=balance,threshold_table=threshold_table,
        validation_scores=validation_scores)
    if frozen_preparation is not None:package['deployment_preparation']=frozen_preparation
    from sklearn.metrics import average_precision_score
    val_y=meta.loc[masks['validation'],'label'].astype(int)
    selection_hash=hashlib.sha256(pd.util.hash_pandas_object(meta.loc[masks['validation'],['label','label_group']],index=True).to_numpy().tobytes()).hexdigest()
    package['validation_selection']=dict(index_hash=selection_hash,rows=len(val_y),positive=int(val_y.sum()),
        average_precision=float(average_precision_score(val_y,validation_scores)) if val_y.nunique()==2 else None)
    times=pd.DatetimeIndex(meta.index.get_level_values('time'))
    vquery=meta.eligible&(times>=pd.Timestamp(boundaries['train_end']))&(times<pd.Timestamp(boundaries['validation_end'])-pd.Timedelta(hours=options['window_hours']))
    vp=meta.loc[vquery].copy()
    vp['score']=model.predict_proba(Pool(direct_native(features.loc[vquery],columns,categorical),cat_features=categorical),thread_count=options['threads'])[:,1]
    package['validation_predictions']=vp
    package['alarm_step']=(cfg or {}).get('step','1h')
    if options['alarm_selection'] and contexts is not None:
        package['alarm_thresholds'],package['alarm_search']=direct_alarm_thresholds(vp,contexts,options,package['alarm_step'],
            boundaries['validation_end'],boundaries['train_end'],boundaries['validation_end'])
    package['thresholds'],_=direct_thresholds(package)
    joblib.dump(package,cache,compress=3)
    # Predict known test examples and the unlabelled tail without using their outcomes.
    inference=meta.eligible&(times>=pd.Timestamp(boundaries['validation_end']))
    if options['test_end']:inference&=times<pd.Timestamp(options['test_end'])
    test_scores=(model.predict_proba(Pool(direct_native(features.loc[inference],columns,categorical),
        cat_features=categorical),thread_count=options['threads'])[:,1] if inference.any() else np.array([]))
    predictions=meta.loc[inference].copy();predictions['score']=test_scores
    predictions['prediction_05']=test_scores>=.5
    predictions['prediction_tuned']=(test_scores>=threshold) if np.isfinite(threshold) else pd.NA
    predictions['tuned_threshold_available']=bool(np.isfinite(threshold))
    predictions['evaluation_eligible']=masks['test'].reindex(predictions.index).fillna(False)
    predictions['model_available_at']=pd.Timestamp(package['available_at'])
    rows=[]
    for name,labels,scores in [('validation',meta.loc[masks['validation'],'label'],validation_scores),
        ('test',predictions.loc[predictions.evaluation_eligible,'label'],predictions.loc[predictions.evaluation_eligible,'score'])]:
        for label,rule in package['thresholds'].items():
            rows.append(dict(partition=name,decision=label,**direct_metrics(labels,scores,rule['threshold'])))
    tested=predictions.loc[predictions.evaluation_eligible]
    for name,values in [('constant_train_prevalence',np.full(len(tested),package['train_prevalence'])),
        ('random_uniform_guess',np.random.default_rng(options['seed']).random(len(tested))),('always_alarm',np.ones(len(tested)))]:
        rows.append(dict(partition='test',decision=name,**direct_metrics(tested.label,values,.5)))
    package.update(predictions=predictions,metrics=pd.DataFrame(rows),
        importance=pd.DataFrame(dict(feature=columns,importance=model.get_feature_importance(thread_count=options['threads']))).sort_values('importance',ascending=False),
        learning_curve=model.get_evals_result())
    joblib.dump(package,cache,compress=3)
    _direct_save_reports(package,folder)
    return package,masks,partitions,balance


def _direct_save_reports(package,folder):
    folder=Path(folder)
    for name,key in [('metrics','metrics'),('split_summary','split_summary'),('balance_summary','balance_summary'),
        ('threshold_search','threshold_table'),('feature_importance','importance')]:
        package[key].to_csv(folder/(name+'.csv'),index=False,encoding='utf-8-sig')
    package['schema'].to_csv(folder/'feature_schema.csv',index=False,encoding='utf-8-sig')
    package['predictions'].to_parquet(folder/'test_predictions.parquet')
    if 'validation_predictions' in package:package['validation_predictions'].to_parquet(folder/'validation_predictions.parquet')
    if 'alarm_search' in package:package['alarm_search'].to_csv(folder/'alarm_threshold_search.csv',index=False,encoding='utf-8-sig')
    (folder/'settings.json').write_text(json.dumps(dict(options=package['options'],boundaries=package['boundaries'],
        threshold=package['threshold'],threshold_status=package['threshold_status'],thresholds=package.get('thresholds'),
        feature_columns=package['columns'],categorical_columns=package['categorical'],
        dropped_columns=package['dropped_columns'],score_status=package['score_status'],
        available_at=package['available_at'],alarm_step=package.get('alarm_step','1h'),
        train_prevalence=package.get('train_prevalence'),validation_selection=package.get('validation_selection')),
        ensure_ascii=False,indent=2),encoding='utf-8')


def direct_predict(package,features):
    from catboost import Pool
    times=(pd.DatetimeIndex(features.index.get_level_values('time'))
        if isinstance(features.index,pd.MultiIndex) else pd.DatetimeIndex(features.index))
    if len(times) and times.min()<pd.Timestamp(package['available_at']):
        raise ValueError('Модель ещё не была доступна на запрошенную дату: '+package['available_at'])
    x=direct_native(features,package['columns'],package['categorical'])
    score=package['model'].predict_proba(Pool(x,cat_features=package['categorical']),
        thread_count=package['options']['threads'])[:,1]
    available=np.isfinite(package['threshold'])
    return pd.DataFrame(dict(score=score,prediction=(score>=package['threshold']) if available else pd.NA,
                             prediction_available=available),index=features.index)


def direct_event_report(package,contexts,threshold=None):
    from monitoring_v3.event_registry import event_type_at
    rows=[];predictions=package['predictions'];left=pd.Timestamp(package['available_at'])
    horizon=pd.Timedelta(hours=package['options']['horizon_hours'])
    for unit,context in contexts.items():
        part=predictions.xs(unit,level='unit') if unit in predictions.index.get_level_values('unit') else pd.DataFrame()
        right=context['frame'].index[-1]
        for event in context['events'].itertuples():
            kind,known=event_type_at(event,right)
            if kind!='Авария' or pd.isna(event.stop) or not left<=event.stop<=right:continue
            if package['options'].get('test_end') and event.stop>=pd.Timestamp(package['options']['test_end']):continue
            window=part.loc[(part.index>=event.stop-horizon)&(part.index<event.stop)] if len(part) else part
            window=window.loc[window.evaluation_eligible&window.label.eq(1)] if len(window) else window
            if len(window):window=window.loc[window.label_group.astype(str).str.endswith(str(event.event_id))]
            warnings=window.loc[window.score.ge(package['threshold'] if threshold is None else threshold)] if len(window) else window
            rows.append(dict(unit=unit,event_id=event.event_id,stop=event.stop,
                scored_points=len(window),window_complete_in_test=event.stop-horizon>=left,
                detected=bool(len(warnings)),first_warning=warnings.index[0] if len(warnings) else pd.NaT,
                lead_hours=float((event.stop-warnings.index[0])/pd.Timedelta('1h')) if len(warnings) else np.nan))
    return pd.DataFrame(rows,columns=['unit','event_id','stop','scored_points','window_complete_in_test','detected','first_warning','lead_hours'])


def direct_table_html(title,frame,opened=False):
    if frame is None or frame.empty:content='<p>Нет данных.</p>'
    else:content='<div style="overflow:auto;max-height:520px">'+frame.to_html(
        index=False,escape=True,na_rep='—',float_format=lambda v:f'{v:.5g}')+'</div>'
    return f'<details{" open" if opened else ""}><summary><b>{escape(title)}</b></summary>{content}</details>'


def _direct_figure_html(figure,path):
    """Always embed a real PNG, even with an Agg/non-inline matplotlib backend."""
    import matplotlib.pyplot as plt
    buffer=BytesIO()
    try:
        figure.savefig(buffer,format='png',dpi=120,bbox_inches='tight')
        data=buffer.getvalue();Path(path).write_bytes(data)
        return '<img style="max-width:100%;height:auto" src="data:image/png;base64,'+base64.b64encode(data).decode('ascii')+'" />'
    finally:plt.close(figure)


def direct_nonzero_importance(package):
    importance=package['importance'].copy()
    values=pd.to_numeric(importance.importance,errors='coerce')
    return importance.loc[values.notna()&values.ne(0)].sort_values('importance',ascending=False)


def direct_report_summary(package,contexts,model_name='model'):
    rules,_=direct_thresholds(package);rows=[];predictions=package['predictions']
    for unit in predictions.index.get_level_values('unit').unique():
        same=predictions.index.get_level_values('unit')==unit;part=predictions.loc[same]
        known=part.evaluation_eligible;test=part.loc[known]
        if test.empty:
            rows.append(dict(model=model_name,unit=unit,decision='no_evaluable_test',rows=0));continue
        for key,rule in rules.items():
            if rule.get('available',True) is False:
                rows.append(dict(model=model_name,unit=unit,decision=key,rows=0,
                    threshold_status=rule['status'],excluded_threshold_rows=len(test)));continue
            threshold=rule['threshold'];decision=None
            if rule.get('category')=='alarm_policy':
                event_metrics,stream,_=direct_alarm_evaluate(part,{unit:contexts[unit]} if unit in contexts else {},
                    package['options'],threshold,package.get('alarm_step','1h'),
                    start=package['available_at'],end=package['options'].get('test_end'))
                decision=stream.loc[test.index,'alarm_active'].to_numpy()
            else:
                events=direct_event_report(package,{unit:contexts[unit]} if unit in contexts else {},threshold)
                event_metrics=dict(events=len(events),detected_events=int(events.detected.sum()) if len(events) else 0,
                    event_recall=float(events.detected.mean()) if len(events) else None,
                    unscored_events=int(events.scored_points.eq(0).sum()) if len(events) else 0,
                    median_lead_hours=float(events.lead_hours.median()) if len(events) and events.lead_hours.notna().any() else None)
            rows.append(dict(model=model_name,unit=unit,decision=key,threshold_status=rule['status'],
                **direct_metrics(test.label,test.score,threshold,decision),**event_metrics))
        unit_seed=int(hashlib.sha256(str(unit).encode()).hexdigest()[:8],16)
        for label,values in [('random_uniform_guess',np.random.default_rng(int(package['options']['seed'])+unit_seed).random(len(test))),
            ('constant_train_prevalence',np.full(len(test),package['train_prevalence'])),('always_alarm',np.ones(len(test)))]:
            rows.append(dict(model=model_name,unit=unit,decision=label,**direct_metrics(test.label,values,.5)))
    return pd.DataFrame(rows)


def direct_plot_segments(ax,index,values,step='1h',**kwargs):
    """A missing/stopped interval must never look like an observed score trend."""
    series=pd.Series(np.asarray(values),index=pd.DatetimeIndex(index)).sort_index()
    if series.empty:return []
    grid=pd.date_range(series.index[0],series.index[-1],freq=step)
    series=series.reindex(grid)
    return ax.plot(series.index,series.to_numpy(),**kwargs)


def direct_report_html(package,contexts,folder,model_name=None):
    """Folded per-boiler test reports; thresholds are selected on validation only."""
    from sklearn.metrics import classification_report,ConfusionMatrixDisplay,PrecisionRecallDisplay,RocCurveDisplay
    import matplotlib.pyplot as plt
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
    model_name=model_name or folder.name
    rules,curve=direct_thresholds(package)
    predictions=package['predictions'];rows=[];blocks=[];event_tables=[]
    number=lambda v:'—' if v is None or not np.isfinite(v) else f'{v:.4f}'
    importance=direct_nonzero_importance(package)
    # Group-level optional diagnostics are rendered once, never a validation metrics table.
    fig,axes=plt.subplots(1,2,figsize=(13,3.5))
    for name in ('precision','recall','f05','f1','f2'):axes[0].plot(curve.threshold,curve[name],label=name)
    for rule in list(rules.values())[1:]:
        if np.isfinite(rule['threshold']):axes[0].axvline(rule['threshold'],ls=':',alpha=.5)
    axes[0].set_title('Подбор порогов: только validation');axes[0].legend(fontsize=8)
    for name,values in package.get('learning_curve',{}).items():
        if 'Logloss' in values:axes[1].plot(values['Logloss'],label=name)
    axes[1].set_title('Logloss по итерациям');axes[1].legend();fig.tight_layout()
    diagnostics=_direct_figure_html(fig,folder/'validation_threshold.png')
    importance_html=direct_table_html('Важность признаков: только ненулевые',importance)
    if len(importance):
        top=importance.head(25).sort_values('importance')
        fig,ax=plt.subplots(figsize=(12,8));ax.barh(top.feature,top.importance)
        ax.set_title('25 наиболее важных признаков · общая модель '+model_name)
        fig.tight_layout();importance_html+='<details><summary>График важности признаков</summary>'+_direct_figure_html(fig,folder/'feature_importance.png')+'</details>'
    groups=package.get('balance_summary',pd.DataFrame())
    counts={int(r.label):int(r.rows) for r in groups.itertuples()} if len(groups) else {}
    balanced=package['options'].get('balance_train',True)
    training_note=('Сохранённая модель обучалась с балансировкой. Для обучения без неё повторите пункт 5.'
        if balanced else 'Обучение без балансировки и классовых весов; validation и test также не балансируются.')
    training_note+=f' В фактическом train группы: класс 0 — {counts.get(0,"—")}, класс 1 — {counts.get(1,"—")}.'
    for unit in sorted(predictions.index.get_level_values('unit').unique()):
        part=predictions.xs(unit,level='unit')
        test=part.loc[part.evaluation_eligible]
        if test.empty:
            blocks.append(f'<details><summary><b>{escape(unit)} · {escape(model_name)}</b></summary><p>Нет тестовых точек с известным исходом.</p></details>')
            continue
        y=test.label.astype(int);score=test.score
        ranking=direct_metrics(y,score,.5)
        body=f'<p>{escape(training_note)}</p><p><b>Независимый test:</b> {len(test)} часов; '
        body+=f'класс 0: {int(y.eq(0).sum())}; класс 1: {int(y.eq(1).sum())}.</p>'
        body+=f'<p>Входов модели: {len(package["columns"])}; деревьев: {getattr(package.get("model"),"tree_count_","—")}; '
        body+='доступна с '+escape(str(package['available_at']))+'.</p>'
        body+=f'<p><b>ROC AUC:</b> {number(ranking["roc_auc"])} · случайный ориентир: 0.5000<br>'
        body+=f'<b>Average Precision:</b> {number(ranking["average_precision"])} · случайный ориентир: {y.mean():.4f} (доля положительных).</p>'
        body+='<p>Пороги подбираются на validation модели; данные этого теста в подборе не участвовали. Если подбор невозможен, причина и fallback подписаны под вариантом.</p>'
        if y.nunique()<2:body+='<p><b>В тесте один класс: ROC AUC и AP здесь не оцениваются.</b></p>'
        opened_category=None
        for key,rule in rules.items():
            category=rule.get('category','positive_class')
            extra=category in ('both_classes','alarm_policy')
            if opened_category and opened_category!=category:body+='</details>';opened_category=None
            if extra and opened_category!=category:
                title='Дополнительно: F по обоим классам (macro)' if category=='both_classes' else 'Дополнительно: устойчивые эпизоды и бюджет ложных тревог'
                body+='<details><summary><b>'+title+'</b></summary>';opened_category=category
            if rule.get('available',True) is False:
                body+='<h4>'+escape(rule['title'])+'</h4><p>Порог не подобран: '+escape(rule['status'])+'. Метрики этого решения не выдаются за проверку рабочего порога.</p>'
                continue
            threshold=rule['threshold'];pred=score.ge(threshold)
            policy_events=None;policy_metrics=None
            if category=='alarm_policy':
                unit_predictions=predictions.loc[predictions.index.get_level_values('unit')==unit]
                policy_metrics,stream,policy_events=direct_alarm_evaluate(unit_predictions,{unit:contexts[unit]} if unit in contexts else {},
                    package['options'],threshold,package.get('alarm_step','1h'),start=package['available_at'],end=package['options'].get('test_end'))
                pred=stream.xs(unit,level='unit').alarm_active.reindex(test.index)
            report=classification_report(y,pred,labels=[0,1],target_names=['Нет аварии','Авария'],digits=3,zero_division=0)
            fig,ax=plt.subplots(figsize=(4.5,3.7))
            ConfusionMatrixDisplay.from_predictions(y,pred,labels=[0,1],display_labels=['Нет аварии','Авария'],ax=ax,colorbar=False)
            ax.set_title(f'{unit} · порог {threshold:.5f}');fig.tight_layout()
            matrix=_direct_figure_html(fig,folder/(unit+'__'+key+'_confusion.png'))
            body+=f'<h4>{escape(rule["title"])} · порог {threshold:.6f}</h4>'
            if any(word in rule['status'] for word in ('fallback','not_evaluable','no_events_detected')):
                body+='<p><b>Ограничение подбора:</b> '+escape(rule['status'])+'. Смотрите наличие обоих классов/аварий и допустимого фона в validation.</p>'
            body+='<div style="display:flex;gap:20px;align-items:center;flex-wrap:wrap">'
            body+='<pre style="overflow:auto">'+escape(report)+'</pre><div style="max-width:480px">'+matrix+'</div></div>'
            if policy_metrics is not None:
                rate=policy_metrics['false_alarms_per_720h'];fraction=policy_metrics['background_alarm_fraction']
                body+='<p><b>Эпизоды на test:</b> ложных уведомлений '+str(policy_metrics['false_notifications'])
                body+='; на 720 фоновых часов '+number(rate)+'; доля фона в тревоге '+number(fraction)+'. '
                body+='Ограничения validation: '+number(package['options']['alarm_budget'])+' / 720 ч и '
                body+=number(package['options']['alarm_background_fraction'])+' доли фона. На test они проверяются заново, а не гарантируются.</p>'
            events=(policy_events if policy_events is not None else direct_event_report(package,{unit:contexts[unit]} if unit in contexts else {},threshold))
            event_tables.append(events.assign(decision=key,threshold=threshold))
            metric=direct_metrics(y,score,threshold)
            metric.update(model=model_name,unit=unit,decision=key,
                events=len(events),detected_events=int(events.detected.sum()) if len(events) else 0,
                event_recall=float(events.detected.mean()) if len(events) else None,
                unscored_events=int(events.scored_points.eq(0).sum()) if len(events) else 0,
                median_lead_hours=float(events.lead_hours.median()) if len(events) and events.lead_hours.notna().any() else None)
            rows.append(metric)
            if len(events):
                body+=f'<p>Аварий обнаружено: {metric["detected_events"]}/{len(events)}; без пригодных точек: {metric["unscored_events"]}.</p>'
                body+=direct_table_html('Отдельные аварии · '+rule['title'],events)
        if opened_category:body+='</details>'
        # A deterministic fair-coin baseline and a constant training-prior baseline.
        unit_seed=int(hashlib.sha256(str(unit).encode()).hexdigest()[:8],16)
        random_scores=np.random.default_rng(int(package['options']['seed'])+unit_seed).random(len(test))
        for label,values in [('random_uniform_guess',random_scores),
            ('constant_train_prevalence',np.full(len(test),package['train_prevalence']))]:
            rows.append(dict(model=model_name,unit=unit,decision=label,**direct_metrics(y,values,.5)))
        body+='<p><b>Baseline:</b> в итоговой таблице есть случайное угадывание '
        body+='(равномерные случайные баллы, порог 0.5, фиксированный seed) и постоянный прогноз по доле аварий в train.</p>'
        if y.nunique()==2:
            fig,axes=plt.subplots(1,2,figsize=(12,3.8))
            PrecisionRecallDisplay.from_predictions(y,score,ax=axes[0],name='CatBoost')
            axes[0].axhline(y.mean(),ls='--',color='grey',label='Случайный ориентир');axes[0].legend()
            RocCurveDisplay.from_predictions(y,score,ax=axes[1],name='CatBoost')
            axes[1].plot([0,1],[0,1],'--',color='grey',label='Случайный ориентир');axes[1].legend()
            fig.tight_layout();body+=_direct_figure_html(fig,folder/(unit+'__test_metrics.png'))
        fig,ax=plt.subplots(figsize=(14,3.8));direct_plot_segments(ax,part.index,part.score,package.get('alarm_step','1h'),lw=.7,label='Балл риска')
        for key,rule in rules.items():
            if rule.get('category') not in ('both_classes','alarm_policy') and np.isfinite(rule['threshold']):
                ax.axhline(rule['threshold'],ls='--',lw=1,alpha=.8,label=rule['title'])
        events=direct_event_report(package,{unit:contexts[unit]} if unit in contexts else {})
        for stop in events.stop:ax.axvline(stop,color='darkred',alpha=.6)
        unknown=part.loc[~part.evaluation_eligible]
        if len(unknown):ax.scatter(unknown.index,unknown.score,s=3,color='grey',label='Исход неизвестен / исключён')
        ax.set_ylim(-.02,1.02);ax.set_title(unit+' · test и последующая история · балл риска')
        ax.legend(fontsize=7,ncol=2);fig.tight_layout()
        body+=_direct_figure_html(fig,folder/(unit+'_risk.png'))
        blocks.append(f'<details class="direct-boiler"><summary style="cursor:pointer;font-size:17px"><b>{escape(unit)} · {escape(model_name)} · '
            f'ROC AUC {number(ranking["roc_auc"])} · AP {number(ranking["average_precision"])}</b></summary>{body}</details>')
    blocks.append('<details><summary><b>Дополнительно · '+escape(model_name)+'</b></summary>'+
        '<details><summary>Validation: пороги и кривая обучения</summary>'+diagnostics+'</details>'+importance_html+'</details>')
    summary=direct_report_summary(package,contexts,model_name)
    if len(summary):
        first=['unit','model','decision','threshold','rows','negative','positive','precision','recall','f05','f1','f2',
            'roc_auc','random_roc_auc','average_precision','random_average_precision']
        summary=summary[[c for c in first if c in summary]+[c for c in summary if c not in first]]
    summary.to_csv(folder/'test_summary_by_boiler.csv',index=False,encoding='utf-8-sig')
    if event_tables:pd.concat(event_tables,ignore_index=True).to_csv(folder/'test_events.csv',index=False,encoding='utf-8-sig')
    importance.to_csv(folder/'feature_importance_nonzero.csv',index=False,encoding='utf-8-sig')
    html='<div class="direct-risk-report">'+''.join(blocks)+'</div>'
    (folder/'report.html').write_text('<meta charset="utf-8">'+html,encoding='utf-8')
    return html,summary


def direct_standard_report(package,contexts,folder,model_name=None):
    from IPython.display import display,HTML
    html,summary=direct_report_html(package,contexts,folder,model_name)
    display(HTML(html))
    return summary


In [ ]:
"""Sequential, disk-cached direct-risk experiments and rolling temporal checks."""
import gc
import hashlib
import json
from pathlib import Path
import traceback
import numpy as np
import pandas as pd

try:
    direct_options
except NameError:
    from direct_risk import (direct_options,direct_features,direct_fit,direct_thresholds,
        direct_report_summary,direct_standard_report,direct_metrics,direct_alarm_evaluate,
        direct_table_html,direct_predict,direct_report_html,direct_split,direct_plot_segments,_direct_figure_html)


def _experiment_json(path,value):
    path=Path(path);temporary=path.with_suffix('.tmp.json')
    temporary.write_text(json.dumps(value,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
    temporary.replace(path)


def trim_direct_contexts(contexts):
    """Free large legacy regression-only matrices after the source audit is saved."""
    for c in contexts.values():
        for field in ('features','quality','input_quality','changes'):
            c.pop(field,None)
    gc.collect()


def prepare_direct_cache(contexts,cfg,options,root,progress=print):
    """One float32 matrix per boiler, eligible hours only; never a fleet-wide union."""
    root=Path(root);root.mkdir(parents=True,exist_ok=True);entries={};summaries=[]
    feature_keys=('window_hours','lags_hours','trends_hours','delta_hours','switch_hours',
        'include_since_start','include_startup','minimum_numeric_coverage')
    for unit,c in contexts.items():
        signature=hashlib.sha256(json.dumps(dict(contract=4,options={k:options[k] for k in feature_keys},
            step=cfg['step'],roles=c['roles'],units=c.get('declared_units',{}),
            reference=str(c.get('preparation_reference_end')),threshold_source=c.get('threshold_source'),
            evidence=c.get('regime_spec',{})),sort_keys=True,default=str).encode())
        for frame in (c['cleaned'][[s for s,r in c['roles'].items() if r in ('load','condition','target')]],
                      c['frame'][[s for s,r in c['roles'].items() if r=='state']],c['regime'][['mode','since_start_hours']]):
            signature.update(pd.util.hash_pandas_object(frame,index=True).to_numpy().tobytes())
        folder=root/unit/signature.hexdigest()[:24];folder.mkdir(parents=True,exist_ok=True)
        marker=folder/'complete.json'
        required=['features.parquet','metadata.parquet','schema.csv','profile.csv','preview.parquet','tail.parquet']
        if marker.exists() and all((folder/name).exists() for name in required):
            row=json.loads(marker.read_text(encoding='utf-8'));progress(unit+': признаки восстановлены из кеша')
        else:
            progress(unit+': подготовка признаков только пригодных часов')
            x,meta,schema=direct_features(c,cfg,options,eligible_only=True)
            profile=schema.merge(pd.DataFrame(dict(feature=x.columns,dtype=x.dtypes.astype(str).to_numpy(),
                missing_fraction=x.isna().mean().to_numpy(),unique_values=x.nunique(dropna=True).to_numpy())),on='feature')
            x.to_parquet(folder/'features.parquet');meta.to_parquet(folder/'metadata.parquet')
            schema.to_csv(folder/'schema.csv',index=False,encoding='utf-8-sig')
            profile.to_csv(folder/'profile.csv',index=False,encoding='utf-8-sig')
            x.head(5).to_parquet(folder/'preview.parquet')
            x.tail(1).to_parquet(folder/'tail.parquet')
            row=dict(unit=unit,source_rows=len(meta),eligible_rows=len(x),features=len(x.columns),
                numeric_sensors=sum(r in ('target','condition','load') for r in c['roles'].values()),
                categorical_sensors=sum(r=='state' for r in c['roles'].values()),
                matrix_mb=float(x.memory_usage(deep=True).sum()/1024**2),folder=str(folder.resolve()))
            _experiment_json(marker,row);del x,meta,schema,profile;gc.collect()
        entries[unit]={**row,'folder':str(folder.resolve())};summaries.append(entries[unit])
    return entries,pd.DataFrame(summaries)


def cached_direct_labels(context,entry,cfg,options):
    from monitoring_v3.interpretation import risk_labels
    c=context;folder=Path(entry['folder'])
    signature=hashlib.sha256(json.dumps(dict(horizon=options['horizon_hours'],lead=options['min_lead_hours'],
        testing=hashlib.sha256(pd.util.hash_pandas_object(c['frame'].get('_Тип',pd.Series('',index=c['frame'].index)),index=True).to_numpy().tobytes()).hexdigest(),
        events=c['events'].to_json(date_format='iso'),step=cfg['step']),sort_keys=True).encode()).hexdigest()[:20]
    target=folder/f'labels_{options["horizon_hours"]}h_{signature}.parquet'
    audit_path=target.with_suffix('.json')
    if target.exists() and audit_path.exists():return pd.read_parquet(target),json.loads(audit_path.read_text(encoding='utf-8'))
    meta=pd.read_parquet(folder/'metadata.parquet');source_rows=len(meta);meta=meta.loc[meta.eligible].copy()
    testing=c['frame'].get('_Тип',pd.Series('',index=c['frame'].index)).reindex(meta.index).isin(['ИСП','ЗРР'])
    as_of=c['frame'].index[-1]
    y,g,audit,known=risk_labels(meta.index[~testing],c['events'],options['horizon_hours'],as_of,as_of,
        observed=c['cleaned'][c['load']].notna(),min_lead_hours=options['min_lead_hours'],
        return_audit=True,return_available_at=True)
    meta['label']=y;meta['label_group']=c['unit']+'::'+g.astype(str)
    meta['label_known_at']=known;meta['source_end']=as_of
    audit.update(unit=c['unit'],horizon_hours=options['horizon_hours'],source_rows=source_rows,
        excluded_testing_rows=int(testing.sum()),
        eligible_rows=len(meta),positive_rows=int(y.eq(1).sum()),negative_rows=int(y.eq(0).sum()),
        positive_events=int(g.loc[y.eq(1)].nunique()))
    meta.to_parquet(target);_experiment_json(audit_path,audit)
    return meta,audit


def experiment_families(contexts,cfg,methods):
    if not methods or set(methods)-{'unit','group'}:raise ValueError('TRAINING_METHODS: список unit / group.')
    families=[]
    for method in dict.fromkeys(methods):
        if method=='unit':
            families.extend(dict(method='unit',name=u,units=[u]) for u in contexts)
        else:
            for group,units in cfg['groups'].items():
                members=[u for u in units if u in contexts]
                if not members:continue
                if len(members)==1 and 'unit' in methods:continue
                families.append(dict(method='group',name=group,units=members))
    return families


def experiment_columns(entries,units,variant):
    if variant not in ('numeric','numeric_states','full'):raise ValueError('FEATURE_VARIANTS: numeric, numeric_states, full.')
    schemas=[pd.read_csv(Path(entries[u]['folder'])/'schema.csv',keep_default_na=False) for u in units]
    common=set(schemas[0].feature)
    for schema in schemas[1:]:common&=set(schema.feature)
    selected=schemas[0].loc[schemas[0].feature.isin(common)].copy()
    for schema in schemas[1:]:
        other=schema.set_index('feature');own=selected.set_index('feature')
        for feature in selected.feature:
            if str(own.loc[feature,'units']).strip().casefold()!=str(other.loc[feature,'units']).strip().casefold():
                raise ValueError('Разные физические единицы общего признака: '+feature)
    if variant=='numeric':selected=selected.loc[selected.feature.str.startswith('num::')|selected.feature.eq('unit')]
    elif variant=='numeric_states':
        selected=selected.loc[~selected.feature.eq('since_start_hours')&~selected.feature.str.endswith('::held_hours')]
    if not selected.feature.str.startswith('num::').any():raise ValueError('Нет общих физических числовых каналов для группы.')
    return selected


def load_experiment_features(entries,units,variant,max_mb=2048):
    schema=experiment_columns(entries,units,variant);columns=list(schema.feature)
    ncat=int(schema.kind.eq('category').sum());nnum=len(schema)-ncat
    estimate=sum(entries[u]['eligible_rows'] for u in units)*(nnum*4+ncat*80+64)/1024**2
    if max_mb is not None and estimate>max_mb:
        raise ValueError(f'Оценка активной матрицы {estimate:.0f} МБ > MAX_FEATURE_MATRIX_MB={max_mb}. '
            'Сократите группу/набор признаков или измените лимит при достаточной памяти.')
    parts={}
    for unit in units:
        p=pd.read_parquet(Path(entries[unit]['folder'])/'features.parquet',columns=columns)
        p.index.name='time';parts[unit]=p
    x=pd.concat(parts,names=['unit','time']).swaplevel().sort_index(kind='stable');x.index.names=['time','unit']
    del parts;gc.collect()
    return x,schema,float(x.memory_usage(deep=True).sum()/1024**2)


def rolling_policy(rolling):
    defaults=dict(adaptive_validation=True,max_validation_days=730,validation_step_days=30,
        min_train_events=2,min_validation_events=2,min_validation_negative_groups=2)
    result={k:rolling.get(k,v) for k,v in defaults.items()}
    if not isinstance(result['adaptive_validation'],bool):raise ValueError('adaptive_validation должен быть bool')
    for key in defaults.keys()-{'adaptive_validation'}:
        value=result[key]
        if isinstance(value,bool) or not isinstance(value,int) or value<1:raise ValueError(key+' должен быть целым >= 1')
    if result['max_validation_days']<rolling.get('validation_days',90):
        raise ValueError('max_validation_days меньше начального validation_days')
    return result


def adaptive_direct_fold(meta,options,fold,rolling):
    """Expand validation into the past; never inspect test outcomes to choose dates."""
    policy=rolling_policy(rolling);b=pd.Timestamp(fold['validation_end']);initial=pd.Timestamp(fold['train_end'])
    t=meta.index.get_level_values('time');past=meta.loc[t<b].copy()
    starts=[initial]
    if policy['adaptive_validation']:
        limit=max(b-pd.Timedelta(days=policy['max_validation_days']),t.min())
        left=initial
        while left>limit:
            left=max(limit,left-pd.Timedelta(days=policy['validation_step_days']));starts.append(left)
    best=None;attempts=[]
    for a in starts:
        opts={**options,'train_end':str(a),'validation_end':str(b),'test_end':fold['test_end'],
              'allow_single_class_evaluation':True}
        try:
            _,summary,_=direct_split(past,opts)
            train=summary.loc[summary.partition.eq('train')].iloc[0]
            val=summary.loc[summary.partition.eq('validation')].iloc[0]
            audit=dict(train_rows=int(train.rows),train_positive=int(train.positive),train_negative=int(train.negative),
                train_events=int(train.positive_events),validation_rows=int(val.rows),validation_positive=int(val.positive),
                validation_negative=int(val.negative),validation_events=int(val.positive_events),
                validation_negative_groups=int(val.negative_groups))
            ready=(audit['train_events']>=policy['min_train_events'] and
                   audit['validation_events']>=policy['min_validation_events'] and
                   audit['validation_negative_groups']>=policy['min_validation_negative_groups'] and
                   audit['train_negative']>0 and audit['validation_negative']>0)
            attempt=dict(train_end=str(a),ready=ready,**audit);attempts.append(attempt)
            if best is None or (audit['validation_events'],audit['validation_negative'])>(best['validation_events'],best['validation_negative']):best=attempt
            if ready:
                return {**fold,'nominal_train_end':str(initial),'train_end':str(a),**audit,
                    'validation_days_actual':float((b-a)/pd.Timedelta('1D')),
                    'split_status':'ready' if a==initial else 'validation_expanded',
                    'split_attempts':len(attempts),'split_reason':''}
        except ValueError as exc:attempts.append(dict(train_end=str(a),ready=False,reason=str(exc)))
    chosen=best or {'train_end':str(initial)}
    return {**fold,**{k:v for k,v in chosen.items() if k!='ready'},'nominal_train_end':str(initial),
        'split_status':'insufficient_past_events','split_attempts':len(attempts),
        'split_reason':f'Нет прошлой выборки с train >= {policy["min_train_events"]} авариями и validation >= '
            f'{policy["min_validation_events"]} авариями / {policy["min_validation_negative_groups"]} группами фона. '
            'Тест не использовался; F-порог не подменяется на 0.5.'}


def rolling_direct_folds(contexts,cfg,rolling):
    allowed={'folds','validation_days','test_days','end',*rolling_policy(rolling)}
    if set(rolling)-allowed:raise ValueError('Неизвестные параметры ROLLING: '+str(set(rolling)-allowed))
    count=int(rolling.get('folds',3));validation=int(rolling.get('validation_days',90));test=int(rolling.get('test_days',90))
    if min(count,validation,test)<1:raise ValueError('Число периодов и длительности должны быть положительными.')
    end=pd.Timestamp(rolling['end']) if rolling.get('end') else max(c['frame'].index[-1] for c in contexts.values())+pd.Timedelta(cfg['step'])
    folds=[]
    for i in range(count):
        test_end=end-pd.Timedelta(days=test*(count-1-i));test_start=test_end-pd.Timedelta(days=test)
        validation_start=test_start-pd.Timedelta(days=validation)
        folds.append(dict(fold=f'fold_{i+1:02d}',train_end=str(validation_start),validation_end=str(test_start),test_end=str(test_end)))
    return folds


def run_direct_suite(contexts,cfg,options,entries,methods,variants,horizons,rolling,root,max_mb=2048,progress=print):
    """Only one active feature matrix, horizon label frame, and fitted model in RAM."""
    import joblib
    root=Path(root);root.mkdir(parents=True,exist_ok=True)
    marker=root/'evaluation_v3.json'
    if not marker.exists():
        import shutil
        backup=root/'previous_reports_before_v3'
        for name in ('experiment_index.csv','fold_metrics.csv','comparison_rolling.csv','rolling_alarm_metrics.csv','validation_choices.csv','selected_models.json'):
            old=root/name
            if old.exists():
                backup.mkdir(exist_ok=True)
                if not (backup/name).exists():shutil.copy2(old,backup/name)
        _experiment_json(marker,dict(contract=3,models_folder='experiments_v3',selection='past_validation_AP',features='telemetry_only_v1'))
    families=experiment_families(contexts,cfg,methods);folds=rolling_direct_folds(contexts,cfg,rolling)
    if not variants or not horizons:raise ValueError('Задайте непустые списки FEATURE_VARIANTS и HORIZONS.')
    if len(set(horizons))!=len(horizons) or any(h<=options['min_lead_hours'] for h in horizons):raise ValueError('Проверьте HORIZONS: уникальные положительные горизонты больше min_lead_hours.')
    records=[];metrics=[];audits=[];plans={}
    def save():
        pd.DataFrame(records).to_csv(root/'experiment_index.csv',index=False,encoding='utf-8-sig')
        (pd.concat(metrics,ignore_index=True) if metrics else pd.DataFrame(columns=['method','name','variant','horizon_hours','fold','unit','decision','rows'])).to_csv(root/'fold_metrics.csv',index=False,encoding='utf-8-sig')
        if audits:pd.DataFrame(audits).drop_duplicates(['unit','horizon_hours']).to_csv(root/'label_audit.csv',index=False,encoding='utf-8-sig')
    for family in families:
        for variant in dict.fromkeys(variants):
            progress(f"{family['method']} · {family['name']} · {variant}: загрузка одной матрицы")
            try:x,schema,mb=load_experiment_features(entries,family['units'],variant,max_mb)
            except ValueError as exc:
                records.append(dict(**{k:v for k,v in family.items() if k!='units'},variant=variant,status='feature_error',reason=str(exc)));save();continue
            progress(f'Активная матрица: {x.shape}; {mb:.1f} МБ (без суммирования горизонтов/моделей)')
            for horizon in horizons:
                base=direct_options({**options,'horizon_hours':horizon,'allow_single_class_evaluation':True})
                parts={}
                for unit in family['units']:
                    progress(f'{unit}: метки горизонта {horizon} ч (кешируются отдельно от признаков)')
                    parts[unit],audit=cached_direct_labels(contexts[unit],entries[unit],cfg,base);audits.append(audit)
                meta=pd.concat(parts,names=['unit','time']).swaplevel().sort_index(kind='stable').reindex(x.index)
                meta.index.names=['time','unit'];del parts
                experiment=f"{family['method']}__{family['name']}__{variant}__{horizon}h"
                for nominal in folds:
                    plan_key=(family['method'],family['name'],horizon,nominal['fold'])
                    if plan_key not in plans:plans[plan_key]=adaptive_direct_fold(meta,base,nominal,rolling)
                    fold=plans[plan_key]
                    folder=root/'experiments_v3'/experiment/fold['fold'];folder.mkdir(parents=True,exist_ok=True)
                    detail=dict(method=family['method'],name=family['name'],units=family['units'],variant=variant,
                        horizon_hours=horizon,fold=fold['fold'],train_end=fold['train_end'],validation_end=fold['validation_end'],test_end=fold['test_end'])
                    record=dict(**{k:v for k,v in detail.items() if k!='units'},experiment=experiment,
                        folder=str(folder.resolve()),matrix_mb=mb,
                        **{k:v for k,v in fold.items() if k not in detail})
                    progress(experiment+' · '+fold['fold'])
                    opts=direct_options({**base,**{k:fold[k] for k in ('train_end','validation_end','test_end')}})
                    t=meta.index.get_level_values('time')
                    if not (meta.eligible&(t>=pd.Timestamp(fold['validation_end']))&(t<pd.Timestamp(fold['test_end']))).any():
                        records.append(dict(**record,status='no_test_observations',reason='Нет рабочих измерений в этом периоде.'));save();continue
                    if fold['split_status']=='insufficient_past_events':
                        records.append(dict(**record,status='insufficient_past_events',reason=fold['split_reason']));save();continue
                    try:
                        package,_,_,_=direct_fit(x,meta,schema,opts,folder,verbose=100,
                            contexts={u:contexts[u] for u in family['units']},cfg=cfg,readiness=rolling_policy(rolling))
                        package['experiment']=detail;joblib.dump(package,folder/'classifier.joblib',compress=3)
                        summary=direct_report_summary(package,{u:contexts[u] for u in family['units']},experiment)
                        for key in ('method','name','variant','horizon_hours','fold','train_end','validation_end','test_end'):summary[key]=detail[key]
                        summary.to_csv(folder/'test_summary_by_boiler.csv',index=False,encoding='utf-8-sig')
                        metrics.append(summary)
                        selection=package['validation_selection']
                        actual=package['split_summary'].set_index('partition')
                        for partition in ('train','validation'):
                            record[partition+'_rows']=int(actual.loc[partition,'rows'])
                            record[partition+'_positive']=int(actual.loc[partition,'positive'])
                            record[partition+'_negative']=int(actual.loc[partition,'negative'])
                            record[partition+'_events']=int(actual.loc[partition,'positive_events'])
                        record['validation_negative_groups']=int(actual.loc['validation','negative_groups'])
                        tested=package['split_summary'].loc[lambda d:d.partition.eq('test')].iloc[0]
                        records.append(dict(**record,status='complete',features=len(package['columns']),trees=package['model'].tree_count_,
                            validation_average_precision=selection['average_precision'],validation_index_hash=selection['index_hash'],
                            test_rows=int(tested.rows),test_positive=int(tested.positive),test_negative=int(tested.negative),
                            test_events=int(tested.positive_events),
                            test_status=('evaluated' if package['predictions'].evaluation_eligible.any() else 'no_known_test_outcomes'),reason=''))
                        del package,summary;gc.collect()
                    except ValueError as exc:
                        records.append(dict(**record,status='insufficient_or_invalid_data',reason=str(exc)))
                        progress('Пропуск с явной причиной: '+str(exc))
                    save()
                del meta;gc.collect()
            del x,schema;gc.collect()
    save()
    select_validation_variants(pd.DataFrame(records),root)
    return pd.DataFrame(records),pd.concat(metrics,ignore_index=True) if metrics else pd.DataFrame()


def select_validation_variants(index,root=None):
    """Choose a feature set per family/horizon/time using only past validation AP."""
    needed={'status','validation_average_precision','validation_index_hash'}
    if not needed.issubset(index):return pd.DataFrame()
    good=index.loc[index.status.eq('complete')&index.validation_average_precision.notna()].copy()
    rows=[]
    for _,group in good.groupby(['method','name','horizon_hours','fold'],sort=False):
        if group.validation_index_hash.nunique()!=1:continue
        order={'numeric':0,'numeric_states':1,'full':2}
        winner=group.assign(_simplicity=group.variant.map(order)).sort_values(
            ['validation_average_precision','_simplicity'],ascending=[False,True]).iloc[0].drop(labels='_simplicity')
        rows.append({**winner.to_dict(),'selected_variant':winner['variant'],
            'selection_status':'selected_on_validation_AP' if len(group)>1 else 'only_available_candidate',
            'candidate_count':len(group)})
    result=pd.DataFrame(rows)
    if root is not None:
        result.to_csv(Path(root)/'validation_choices.csv',index=False,encoding='utf-8-sig')
        if len(result):
            latest=result.sort_values('validation_end').groupby(['method','name','horizon_hours'],sort=False).tail(1)
            _experiment_json(Path(root)/'selected_models.json',latest[[c for c in (
                'method','name','horizon_hours','selected_variant','folder','validation_end','validation_average_precision',
                'selection_status') if c in latest]].to_dict('records'))
        else:_experiment_json(Path(root)/'selected_models.json',[])
    return result


def aggregate_direct_folds(metrics):
    """Sum disjoint test confusions, but never pool scores of different models for AUC/AP."""
    if metrics.empty:return pd.DataFrame()
    keys=['method','name','variant','horizon_hours','unit','decision'];rows=[]
    good=metrics.loc[pd.to_numeric(metrics.rows,errors='coerce').gt(0)]
    for key,g in good.groupby(keys,dropna=False,sort=False):
        row=dict(zip(keys,key));row.update(folds=g.fold.nunique(),rows=int(g.rows.sum()),
            positive=int(g.positive.sum()),negative=int(g.negative.sum()))
        tp,fp,fn,tn=(float(g[c].sum()) for c in ('true_positive','false_positive','false_negative','true_negative'))
        row.update(precision=tp/(tp+fp) if tp+fp else None,recall=tp/(tp+fn) if tp+fn else None,
            specificity=tn/(tn+fp) if tn+fp else None,predicted_positive_fraction=(tp+fp)/row['rows'])
        for beta,name in ((.5,'f05'),(1,'f1'),(2,'f2')):
            pos=(1+beta**2)*tp/((1+beta**2)*tp+beta**2*fn+fp) if tp+fp+fn else 0.
            neg=(1+beta**2)*tn/((1+beta**2)*tn+beta**2*fp+fn) if tn+fp+fn else 0.
            row[name]=pos;row['macro_'+name]=(pos+neg)/2
        for column in ('roc_auc','average_precision','random_average_precision'):
            values=pd.to_numeric(g[column],errors='coerce');row[column+'_mean']=float(values.mean())
            row[column+'_min']=float(values.min());row[column+'_max']=float(values.max())
        valid_ap=pd.to_numeric(g.average_precision,errors='coerce').notna()
        row['rankable_folds']=int(valid_ap.sum())
        row['random_average_precision_mean']=pd.to_numeric(g.loc[valid_ap,'random_average_precision'],errors='coerce').mean()
        row['ap_lift_mean']=(pd.to_numeric(g.loc[valid_ap,'average_precision'],errors='coerce')/
            pd.to_numeric(g.loc[valid_ap,'random_average_precision'],errors='coerce').replace(0,np.nan)).mean()
        row['event_recall_fold_mean']=pd.to_numeric(g.get('event_recall',pd.Series(dtype=float)),errors='coerce').mean()
        rows.append(row)
    return pd.DataFrame(rows)


def direct_rolling_alarm_summary(index,contexts,root):
    """Run the notification state once over all disjoint causal fold predictions.

    Thresholds may change at fold boundaries; the persistence/repeat state does
    not restart there. No optimisation is performed on these test predictions.
    """
    rows=[]
    complete=index.loc[index.status.eq('complete')]
    for experiment,group in complete.groupby('experiment',sort=False):
        pieces=[];options=None;rules={};step='1h';start=None;end=None
        for record in group.sort_values('validation_end').itertuples():
            folder=Path(record.folder)
            settings=json.loads((folder/'settings.json').read_text(encoding='utf-8'))
            options=settings['options'];step=settings.get('alarm_step','1h')
            p=pd.read_parquet(folder/'test_predictions.parquet')
            time=pd.DatetimeIndex(p.index.get_level_values('time'))
            if not pd.to_datetime(p.model_available_at).le(time).all():raise ValueError('Попытка использовать модель раньше её доступности.')
            for key,rule in (settings.get('thresholds') or {}).items():
                if rule.get('category')=='alarm_policy':
                    p['threshold::'+key]=rule['threshold'];rules[key]=rule
            pieces.append(p)
            a=pd.Timestamp(record.validation_end);b=pd.Timestamp(record.test_end)
            start=a if start is None else min(start,a);end=b if end is None else max(end,b)
        if not pieces:continue
        predictions=pd.concat(pieces).sort_index()
        if predictions.index.has_duplicates:raise ValueError('Тестовые периоды пересеклись: '+experiment)
        for key in rules:
            for unit in predictions.index.get_level_values('unit').unique():
                p=predictions.loc[predictions.index.get_level_values('unit')==unit]
                metric,_,events=direct_alarm_evaluate(p,{unit:contexts[unit]} if unit in contexts else {},options,
                    p['threshold::'+key],step=step,start=start,end=end)
                first=group.iloc[0]
                rows.append(dict(method=first.method,name=first['name'],variant=first.variant,horizon_hours=first.horizon_hours,
                    unit=unit,decision=key,**{'rolling_'+k:v for k,v in metric.items()}))
        del predictions,pieces;gc.collect()
    result=pd.DataFrame(rows)
    result.to_csv(Path(root)/'rolling_alarm_metrics.csv',index=False,encoding='utf-8-sig')
    return result


def render_direct_suite(index,contexts,root,report_folds='none'):
    """Read/render one saved model at a time; no training, no PACKAGES collection."""
    from IPython.display import display,HTML
    import joblib
    index=index.copy();root=Path(root)
    # A relocated output folder must not keep referring to another computer's drive.
    if 'folder' in index:
        for i,row in index.iterrows():
            if Path(str(row.folder)).exists():continue
            for sub in ('experiments_v2','experiments'):
                candidate=root/sub/str(row.experiment)/str(row.fold)
                if candidate.exists():index.loc[i,'folder']=str(candidate);break
    good=index.loc[index.status.eq('complete')].copy() if 'status' in index else pd.DataFrame()
    if good.empty:
        display(HTML(direct_table_html('Нет готовых моделей: причины каждого пропуска',index,opened=True)))
        return pd.DataFrame(),pd.DataFrame()
    choices=select_validation_variants(index,root)
    automatic=pd.DataFrame()
    if len(choices):
        automatic=choices.copy();automatic['source_variant']=automatic.variant;automatic['variant']='AUTO'
        automatic['experiment']=automatic.apply(lambda r:f'{r.method}__{r["name"]}__AUTO__{r.horizon_hours}h',axis=1)
        display(HTML('<p><b>AUTO:</b> набор признаков выбирается до очередного теста по AP на одинаковой прошлой validation. '
            'При равенстве предпочтение более простому набору: numeric, numeric_states, full. Тест в выборе не участвует.</p>'+
            direct_table_html('Какие признаки выбирались по прошлому',choices.drop(columns=['validation_index_hash'],errors='ignore'))))
    overall=[]
    all_records=pd.concat([automatic,good],ignore_index=True) if len(automatic) else good
    for experiment,group in all_records.groupby('experiment',sort=False):
        html,summary=combined_direct_report(group,contexts,root/'overall_reports'/experiment)
        display(HTML(html));overall.append(summary)
    # Individual-version reports are optional diagnostics, never the main view.
    detail=good
    if report_folds in ('none',None,False):detail=good.iloc[:0]
    elif report_folds=='latest':
        detail=good.sort_values('test_end').groupby('experiment',sort=False).tail(1)
    elif report_folds!='all':
        if not isinstance(report_folds,(list,tuple)):raise ValueError('REPORT_FOLDS: latest, all или список fold_01...')
        detail=good.loc[good.fold.isin(report_folds)]
    for row in detail.itertuples():
        folder=Path(row.folder);package=joblib.load(folder/'classifier.joblib')
        detail=package.get('experiment',{});units=detail.get('units',list(contexts))
        title=f'{row.method} · {row.name} · {row.variant} · {row.horizon_hours} ч · {row.fold}'
        from html import escape
        html,_=direct_report_html(package,{u:contexts[u] for u in units if u in contexts},folder,title)
        display(HTML('<details><summary><b>Диагностика отдельной версии: '+escape(title)+'</b></summary>'+html+'</details>'))
        del package,html;gc.collect()
    target=Path(root)/'fold_metrics.csv'
    metrics=pd.read_csv(target) if target.exists() else pd.DataFrame()
    aggregate=pd.concat(overall,ignore_index=True) if overall else pd.DataFrame()
    aggregate.to_csv(Path(root)/'comparison_rolling.csv',index=False,encoding='utf-8-sig')
    columns=[c for c in ('unit','method','name','variant','horizon_hours','decision','rows','precision','recall','f1','macro_f1',
        'roc_auc_mean','roc_auc_min','roc_auc_max','average_precision_mean','random_average_precision_mean',
        'rolling_events','rolling_detected_events','rolling_false_notifications','rolling_false_alarms_per_720h',
        'rolling_background_alarm_fraction','excluded_threshold_rows') if c in aggregate]
    display(HTML('<h3>Качество моделей за всю последовательную тестовую историю</h3>'+direct_table_html(
        f'Сводные метрики по котлам ({len(aggregate)} строк)',aggregate[columns],opened=len(aggregate)<=50)+
        direct_table_html('Все показатели модели, включая поддержку и матрицы ошибок',aggregate)))
    display(HTML(direct_table_html('Все отдельные периоды (в том числе macro и эпизоды)',metrics)))
    audit_cols=[c for c in ('method','name','variant','horizon_hours','fold','train_end','validation_end','test_end',
        'validation_days_actual','train_rows','train_events','validation_rows','validation_positive','validation_negative',
        'validation_events','test_rows','test_events','test_status','split_status','status','reason') if c in index]
    display(HTML(direct_table_html('Диагностика выборок: часы, аварии и причины пропусков',index[audit_cols])))
    display(HTML(direct_table_html('Статусы всех запусков: причины пропусков',index)))
    return aggregate,metrics


def combined_direct_report(records,contexts,folder):
    """Standard reports on the continuous held-out history; no averaged confusions."""
    from html import escape
    from sklearn.metrics import classification_report,ConfusionMatrixDisplay
    import matplotlib.pyplot as plt
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
    pieces=[];rules={};version_rows=[];importance=[];ranking=[]
    for row in records.sort_values('validation_end').itertuples():
        source=Path(row.folder);settings=json.loads((source/'settings.json').read_text(encoding='utf-8'))
        p=pd.read_parquet(source/'test_predictions.parquet');times=p.index.get_level_values('time')
        if not pd.to_datetime(p.model_available_at).le(times).all():raise ValueError('Модель применена до доступности')
        p['version']=str(row.fold)
        p['baseline_prevalence']=settings.get('train_prevalence',np.nan)
        for key,rule in (settings.get('thresholds') or {}).items():
            threshold=rule.get('threshold');threshold=float(threshold) if threshold is not None else np.inf
            available=rule.get('available',True) and not any(s in rule.get('status','') for s in ('fallback','not_evaluable','threshold_unavailable'))
            p['threshold::'+key]=threshold if available else np.inf;rules[key]=rule
        for unit in p.index.get_level_values('unit').unique():
            tested=p.xs(unit,level='unit').loc[lambda d:d.evaluation_eligible&d.label.notna()]
            values=direct_metrics(tested.label,tested.score,.5)
            ranking.append(dict(unit=unit,version=row.fold,**values))
        pieces.append(p)
        stats=pd.read_csv(source/'split_summary.csv')
        stats['version']=row.fold;stats['selected_variant']=getattr(row,'source_variant',row.variant)
        version_rows.append(stats)
        imp=source/'feature_importance.csv'
        if imp.exists():importance.append(pd.read_csv(imp))
    if not pieces:return '<p>Нет сохранённых тестовых оценок.</p>',pd.DataFrame()
    p=pd.concat(pieces).sort_index()
    if p.index.has_duplicates:raise ValueError('Тестовые часы пересекаются: '+str(folder))
    first=records.iloc[0];options=settings['options'];step=settings.get('alarm_step','1h')
    start=pd.Timestamp(records.validation_end.min());end=pd.Timestamp(records.test_end.max())
    rank=pd.DataFrame(ranking);summaries=[];blocks=[]
    title=f'{first.method} · {first["name"]} · {first.variant} · {first.horizon_hours} ч'
    num=lambda v:'—' if v is None or pd.isna(v) or not np.isfinite(v) else f'{v:.3f}'
    extra={'both_classes':'Дополнительные пороги: оба класса (macro)',
           'alarm_policy':'Подтверждённые тревоги и бюджет фона'}
    for unit in p.index.get_level_values('unit').unique():
        local=p.loc[p.index.get_level_values('unit')==unit].copy();d=local.xs(unit,level='unit')
        known=d.evaluation_eligible.eq(True)&d.label.notna()&d.score.notna()
        y_all=d.loc[known,'label'].astype(int)
        rr=rank.loc[rank.unit.eq(unit)].copy()
        valid_rank=rr.roc_auc.notna() if 'roc_auc' in rr else pd.Series(False,index=rr.index)
        ranks=dict(roc_auc_mean=rr.loc[valid_rank,'roc_auc'].mean() if valid_rank.any() else np.nan,
            roc_auc_min=rr.loc[valid_rank,'roc_auc'].min() if valid_rank.any() else np.nan,
            roc_auc_max=rr.loc[valid_rank,'roc_auc'].max() if valid_rank.any() else np.nan,
            average_precision_mean=rr.loc[valid_rank,'average_precision'].mean() if valid_rank.any() else np.nan,
            random_average_precision_mean=rr.loc[valid_rank,'random_average_precision'].mean() if valid_rank.any() else np.nan,
            rankable_versions=int(valid_rank.sum()),versions=len(rr))
        body=f'<p><b>Вся независимая тестовая история:</b> {start} → {end}; известных часов {len(y_all)}, '
        body+=f'положительных {int(y_all.sum())}, фоновых {int(y_all.eq(0).sum())}. Версий модели: {len(rr)}.</p>'
        support=pd.concat(version_rows,ignore_index=True)
        for partition,label in (('train','Обучение'),('validation','Подбор')):
            g=support.loc[support.partition.eq(partition)]
            body+=f'<p>{label}: {int(g.rows.min())}–{int(g.rows.max())} строк, '
            body+=f'{int(g.positive_events.min())}–{int(g.positive_events.max())} независимых аварий на версию '
            '(для всей модели/группы). Точные даты и состав скрыты в диагностике.</p>'
        body+=f'<p><b>ROC AUC:</b> среднее {num(ranks["roc_auc_mean"])}; диапазон {num(ranks["roc_auc_min"])}–{num(ranks["roc_auc_max"])}; случайно 0.5.<br>'
        body+=f'<b>Average Precision:</b> среднее {num(ranks["average_precision_mean"])}; случайный AP на тех же периодах {num(ranks["random_average_precision_mean"])}. '
        body+='Баллы разных версий не объединены для AUC/AP; отчёты ниже объединяют реальные ошибки.</p>'
        if not len(y_all):body+='<p>Нет часов с известным исходом; обученная модель пока не оценена.</p>'
        category_open=None
        for key,rule in rules.items():
            category=rule.get('category','positive_class')
            if category_open and category_open!=category:body+='</details>';category_open=None
            if category in extra and category_open!=category:
                body+='<details><summary><b>'+extra[category]+'</b></summary>';category_open=category
            threshold=d.get('threshold::'+key,pd.Series(np.inf,index=d.index))
            available=np.isfinite(threshold);take=known&available
            event_metrics={};events=pd.DataFrame()
            if category=='alarm_policy':
                event_metrics,stream,events=direct_alarm_evaluate(local,{unit:contexts[unit]} if unit in contexts else {},options,
                    local['threshold::'+key],step,start=start,end=end)
                pred=stream.xs(unit,level='unit').alarm_active.reindex(d.index)
            else:pred=d.score.ge(threshold)
            y=y_all.reindex(d.index[take]);selected_pred=pred.loc[take]
            body+='<h4>'+escape(rule['title'])+'</h4>'
            if not len(y):
                body+='<p>Нет пригодного порога на оцениваемых часах. Подстановка 0.5 не выполнялась.</p>'
                summaries.append(dict(method=first.method,name=first['name'],variant=first.variant,horizon_hours=first.horizon_hours,
                    unit=unit,decision=key,rows=0,excluded_threshold_rows=int(known.sum()),threshold_status='unavailable',**ranks));continue
            # With per-version thresholds, derive confusion using the actual
            # decisions; ranking always comes from the independent version table.
            metric=direct_metrics(y,d.loc[take,'score'],.5,selected_pred)
            metric.pop('roc_auc',None);metric.pop('average_precision',None);metric.pop('threshold',None)
            metric.update(excluded_threshold_rows=int((known&~available).sum()))
            report=classification_report(y,selected_pred,labels=[0,1],target_names=['Фон','Перед аварией'],digits=3,zero_division=0)
            fig,ax=plt.subplots(figsize=(4.5,3.7))
            ConfusionMatrixDisplay.from_predictions(y,selected_pred,labels=[0,1],display_labels=['Фон','Перед аварией'],ax=ax,colorbar=False)
            ax.set_title('Вся тестовая история');fig.tight_layout()
            body+='<div style="display:flex;gap:18px;flex-wrap:wrap;align-items:center"><pre>'+escape(report)+'</pre>'
            body+=_direct_figure_html(fig,folder/(unit+'__'+key+'_confusion.png'))+'</div>'
            body+='<p>У каждой версии свой порог из прошлого; исключено часов без пригодного порога: '+str(metric['excluded_threshold_rows'])+'.</p>'
            if event_metrics:
                body+=f'<p><b>Уведомления за всю историю:</b> обнаружено {event_metrics["detected_events"]}/{event_metrics["events"]}; '
                body+=f'ложных {event_metrics["false_notifications"]}; на 720 ч фона {num(event_metrics["false_alarms_per_720h"])}; '
                body+=f'доля фона в тревоге {num(event_metrics["background_alarm_fraction"])}; без оценок {event_metrics["unscored_events"]} аварий.</p>'
                body+=direct_table_html('Все аварии и обнаружение',events)
            summaries.append(dict(method=first.method,name=first['name'],variant=first.variant,horizon_hours=first.horizon_hours,
                unit=unit,decision=key,**metric,**ranks,**{'rolling_'+k:v for k,v in event_metrics.items()}))
        if category_open:body+='</details>'
        seed=int(hashlib.sha256(str(unit).encode()).hexdigest()[:8],16)
        baselines={'always_alarm':np.ones(len(y_all)),'random_uniform_guess':np.random.default_rng(seed).random(len(y_all)),
                   'constant_train_prevalence':d.loc[known,'baseline_prevalence'].to_numpy()}
        for key,values in baselines.items():
            if not np.isfinite(values).all():continue
            metric=direct_metrics(y_all,values,.5);metric.pop('roc_auc',None);metric.pop('average_precision',None)
            summaries.append(dict(method=first.method,name=first['name'],variant=first.variant,horizon_hours=first.horizon_hours,
                unit=unit,decision=key,**metric))
        fig,axes=plt.subplots(2,1,figsize=(14,6),sharex=True)
        direct_plot_segments(axes[0],d.index,d.score,step,lw=.7,label='Балл риска')
        for key in ('default_05','best_f2'):
            col='threshold::'+key
            if col in d:direct_plot_segments(axes[0],d.index,d[col].where(np.isfinite(d[col])),step,lw=.9,ls='--',label=key)
        axes[0].set_ylim(-.02,1.02);axes[0].set_ylabel('Балл');axes[0].legend(fontsize=8)
        if 'threshold::alarm_budget_recall' in d:
            _,stream,events=direct_alarm_evaluate(local,{unit:contexts[unit]} if unit in contexts else {},options,
                local['threshold::alarm_budget_recall'],step,start=start,end=end)
            state=stream.xs(unit,level='unit').alarm_active.astype(float)
            direct_plot_segments(axes[1],state.index,state,step,lw=.8,label='Подтверждённая тревога')
            for event in events.itertuples():
                for ax in axes:ax.axvline(event.stop,color='red',alpha=.4)
        axes[1].set_ylabel('Тревога');axes[0].set_title(unit+' · '+title+' · вся тестовая история')
        for ax in axes:ax.grid(alpha=.2)
        fig.tight_layout();body+=_direct_figure_html(fig,folder/(unit+'_whole_history.png'))
        blocks.append('<details class="direct-boiler"><summary><b>'+escape(unit+' · '+title)+'</b></summary>'+body+'</details>')
    diagnostics=direct_table_html('Диагностика отдельных версий: состав выборок',pd.concat(version_rows,ignore_index=True))
    if importance:
        imp=pd.concat(importance).groupby('feature',as_index=False).importance.sum()
        imp['importance']=imp.importance/len(importance);imp=imp.loc[imp.importance.ne(0)].sort_values('importance',ascending=False)
        diagnostics+=direct_table_html('Ненулевая важность: среднее по версиям, отсутствие признака считается нулём',imp)
        if len(imp):
            top=imp.head(25).iloc[::-1];fig,ax=plt.subplots(figsize=(12,8))
            ax.barh(top.feature,top.importance);ax.set_title('Важность входов: среднее по версиям');fig.tight_layout()
            diagnostics+='<details><summary>График важности входов</summary>'+_direct_figure_html(fig,folder/'importance.png')+'</details>'
    html='<details><summary><b>'+escape(title+' · общий результат')+'</b></summary>'+''.join(blocks)+diagnostics+'</details>'
    result=pd.DataFrame(summaries);result.to_csv(folder/'overall_metrics.csv',index=False,encoding='utf-8-sig')
    (folder/'report.html').write_text('<meta charset="utf-8">'+html,encoding='utf-8')
    return html,result


## 1. Все настройки

- `TRAINING_METHODS`: `unit` — отдельный котёл, `group` — общая модель котлов одного
  типа из CFG.groups. Для группы используется пересечение общих признаков с одинаковыми
  физическими единицами. Однокотловая группа не повторяет unit, если выбран и unit.
- `FEATURE_VARIANTS`: `numeric` — числовые каналы и их динамика; `numeric_states` —
  плюс текущие состояния и числа переключений, **без длительности состояний и возраста
  пуска**; `full` — все согласованные признаки, включая эти длительности. Идентификатор
  котла сохраняется для групповых моделей во всех вариантах.
- `HORIZONS`: список, например `[24, 72, 168]`. По умолчанию `[168]`, чтобы первый запуск
  не умножал число обучений втрое. Матрицы не копируются по горизонтам, но время расчёта увеличивается.
- `ROLLING`: число непересекающихся следующих test-периодов, длительности validation/test
  в сутках и необязательный общий конец. Validation начинается с 90 суток и при необходимости
  расширяется назад шагом validation_step_days, максимум до max_validation_days.
  По умолчанию нужны 2 независимые аварии в train и 2 в validation, а также 2 недельные
  группы фона в validation. Эти минимумы настраиваются явно. Выбор дат не использует test.
  Между train/validation/test
  учитываются 24-часовые окна и доступность исходов; одна авария не переходит в обе части.
- `REPORT_FOLDS="none"`: основной общий отчёт без отдельных картинок фолдов.
  `"latest"`, `"all"` или список `['fold_01', ...]` добавляют закрытую диагностику версий.
  Все тестовые часы входят в общий результат независимо от этой настройки.
- `MAX_FEATURE_MATRIX_MB`: ограничение оценки активной матрицы, не всей памяти Python;
  `None` снимает ограничение. `OPTIONS.threads` ограничивает CPU CatBoost.
- `alarm_selection=True` добавляет два подбора: максимум обнаружения аварий и максимум
  событийного F1 при бюджете. `alarm_persist_hours=3` подтверждает превышение после 3 часов;
  `alarm_merge_hours=12` подавляет близкие повторные уведомления. Ограничения по умолчанию:
  не больше 2 ложных уведомлений / 720 фоновых часов и не больше 5% фона в тревоге.
  При отсутствии полезного решения может быть выбран вариант без тревог; это видно в отчёте.
- `macro_thresholds=True`: дополнительные F1/F2/F0.5 как среднее F двух классов с равным
  весом. Обычный F положительного класса тоже штрафует FP, macro дополнительно учитывает класс 0.

Числовой канал: текущее значение, лаги 12/24 ч, min/max/mean/std за 24 ч, тренды 6/24 ч,
изменения 1/6 ч. Категория: значение, наблюдаемая длительность и переключения 6/24 ч.
Временные признаки используют только прошлое и не переносятся через разрывы рабочего участка.


In [ ]:
SOURCE_NOTEBOOK = "02_Обучение.ipynb"
SOURCE_DATA_NOTEBOOK = "05_Классификатор.ipynb"
DATA_SETTINGS = {
    # "groups": {"БКЗ-500": ["К-5", "К-6", "К-7"]},
    # "data_paths": {"К-5": "D:/private/К-5/hourly.parquet"},
    # "run_thresholds": {"К-5": 140.0},
}
OUTPUT = "runs/direct_classifier_comparison"
TRAINING_METHODS = ["unit", "group"]
FEATURE_VARIANTS = ["numeric", "numeric_states", "full"]
HORIZONS = [168]  # Например [24, 72, 168]
ROLLING = {
    "folds": 3, "validation_days": 90, "test_days": 90, "end": None,
    "adaptive_validation": True, "max_validation_days": 730, "validation_step_days": 30,
    "min_train_events": 2, "min_validation_events": 2, "min_validation_negative_groups": 2,
}
REPORT_FOLDS = "none"  # общий отчёт; latest/all — дополнительная диагностика
MAX_FEATURE_MATRIX_MB = 2048
PREVIEW_UNIT = None  # None — первый котёл, например "К-5"

OPTIONS = direct_options({
    "window_hours": 24, "lags_hours": [12, 24], "trends_hours": [6, 24],
    "delta_hours": [1, 6], "switch_hours": [6, 24],
    "include_since_start": True, "include_startup": True,
    "minimum_numeric_coverage": 0.7, "min_lead_hours": 0,
    "train_step_hours": 6, "iterations": 400, "depth": 4,
    "learning_rate": 0.05, "early_stopping_rounds": 50, "threads": 2,
    "seed": 42, "balance_train": False, "event_weights": False,
    "threshold_metric": "f2", "minimum_precision": None,
    "macro_thresholds": True, "alarm_selection": True,
    "alarm_persist_hours": 3, "alarm_merge_hours": 12,
    "alarm_budget": 2.0, "alarm_background_fraction": 0.05,
    "alarm_candidates": 61, "restore": True,
})

def previous_settings(path):
    result = {}
    if not path.exists():
        return result
    book = json.loads(path.read_text(encoding="utf-8-sig"))
    for cell in book.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        try:
            tree = ast.parse("".join(cell["source"]))
        except SyntaxError:
            continue
        for statement in tree.body:
            if not isinstance(statement, ast.Assign):
                continue
            for target in statement.targets:
                if isinstance(target, ast.Name) and target.id in ("SETTINGS", "DATA_SETTINGS", "SENSOR_RULES"):
                    try:
                        result[target.id] = ast.literal_eval(statement.value)
                    except (ValueError, TypeError):
                        print(target.id + " содержит выражения; задайте нужные пути явно в DATA_SETTINGS.")
    return result

PREVIOUS = previous_settings(TRAINING_DIR/SOURCE_NOTEBOOK)
OLD_DATA = previous_settings(TRAINING_DIR/SOURCE_DATA_NOTEBOOK)
CFG = notebook_config({
    **PREVIOUS.get("SETTINGS", {}), **OLD_DATA.get("DATA_SETTINGS", {}),
    **DATA_SETTINGS, "output": OUTPUT,
    "models": ["CatBoost"], "feature_sets": ["conditions"], "scenarios": ["local"],
}, PREVIOUS.get("SENSOR_RULES", {}))
OUT = Path(CFG["output"]).resolve()
OUT.mkdir(parents=True, exist_ok=True)
display(HTML(direct_table_html("Настройки", pd.DataFrame({
    "параметр": list(OPTIONS), "значение": [str(v) for v in OPTIONS.values()]
}))))
print("Группы:", CFG["groups"])
print("Пути из прежнего 05:", bool(OLD_DATA.get("DATA_SETTINGS")))
print("Результаты:", OUT)


## 2. Источники и режимы

Здесь читаются исходные данные и существующая карта каналов. Разметка ремонта
не подаётся во входы. Неизвестные остановы и неполное будущее не становятся нормой.
Начальный опорный период автоматической подготовки не используется как будто пороги
режимов уже были известны. При ручных порогах он может не требоваться.


In [ ]:
CONTEXTS, ISSUES = load_sources(CFG, OUT/"audit", strict=False)
if ISSUES:
    display(HTML(direct_table_html("Проблемы источников", pd.DataFrame(ISSUES), opened=True)))
    raise ValueError("Исправьте DATA_SETTINGS и повторите подготовку.")
if not CONTEXTS:
    raise ValueError("Нет доступных котлов.")
trim_direct_contexts(CONTEXTS)
print("Освобождены вспомогательные матрицы регрессоров; исходные данные и режимы сохранены.")
SOURCE_SUMMARY = pd.DataFrame([
    {"unit": u, "rows": len(c["frame"]), "first": c["frame"].index.min(),
     "last": c["frame"].index.max(), "load": c["load"], "run_threshold": c["threshold"]}
    for u, c in CONTEXTS.items()
])
display(HTML(direct_table_html("Источники", SOURCE_SUMMARY, opened=True)))
for u, c in CONTEXTS.items():
    display(HTML(direct_table_html(u + " · реестр остановов", c["events"])))


## 3. Кеш признаков по котлам

Одновременно строится только один котёл. В файлах признаков хранятся пригодные часы
и float32; строки стоящего котла с тысячами пустых полей не размножаются.
Метки горизонтов считаются отдельно при обучении и кешируются без повторного построения признаков.
Большие таблицы не объединяются по всем котлам; ниже показана сводка и подробности одного котла.


In [ ]:
CACHE, CACHE_SUMMARY = prepare_direct_cache(CONTEXTS, CFG, OPTIONS, OUT/"feature_cache")
CACHE_SUMMARY.to_csv(OUT/"feature_cache_summary.csv", index=False, encoding="utf-8-sig")
display(HTML(direct_table_html("Размеры отдельных матриц (не хранятся вместе в памяти)", CACHE_SUMMARY, opened=True)))
unit = PREVIEW_UNIT or next(iter(CACHE))
folder = Path(CACHE[unit]["folder"])
profile = pd.read_csv(folder/"profile.csv")
display(HTML(direct_table_html(unit + " · признаки, описания, типы и пропуски", profile)))
preview = pd.read_parquet(folder/"preview.parquet")
display(HTML(direct_table_html(unit + " · пример пяти моментов", preview.T.reset_index(names="feature"))))
del profile, preview
gc.collect()


## 4. План запусков и общие временные границы

Число попыток равно числу семейств × наборов признаков × горизонтов × периодов.
Времени потребуется больше, чем одной модели; память не умножается на число запусков.
Завершённые совместимые модели восстанавливаются. При прерывании готовые результаты
остаются на диске, пункт 7 можно выполнять без возврата всей функции обучения.


In [ ]:
FAMILIES = experiment_families(CONTEXTS, CFG, TRAINING_METHODS)
FOLDS = rolling_direct_folds(CONTEXTS, CFG, ROLLING)
PLAN = pd.DataFrame([{
    "method": f["method"], "name": f["name"], "units": ", ".join(f["units"]),
    "variants": len(FEATURE_VARIANTS), "horizons": len(HORIZONS), "folds": len(FOLDS),
} for f in FAMILIES])
print("Всего попыток:", len(FAMILIES)*len(FEATURE_VARIANTS)*len(HORIZONS)*len(FOLDS))
print("Однокотловые group не дублируют unit.")
display(HTML(direct_table_html("План", PLAN, opened=True)))
display(HTML(direct_table_html("Диагностика: даты тестов и начальная validation", pd.DataFrame(FOLDS))))


## 5. Последовательное обучение и проверка

Во всех частях естественный баланс; train прорежен одинаковым шагом времени.
Ни один тест не используется в подборе порога. В отдельном периоде с одним классом
AP/ROC AUC не определены, но ошибки классификации и ложные тревоги учитываются.
Если у короткой истории нет наблюдений в позднем периоде, это явный статус;
истории остальных котлов не обрезаются по самому короткому.

Приоритетные четыре порога — по баллам. Дополнительные macro-варианты тоже по баллам.
Дополнительные бюджетные варианты оцениваются **после подтверждения и подавления повторов**.
Тревога, начавшаяся задолго до предаварийного окна, не получает автоматический зачёт
обнаружения только за то, что продолжала гореть. Неизвестные исходы учитываются отдельно.


In [ ]:
RUN_INDEX, FOLD_METRICS = run_direct_suite(
    CONTEXTS, CFG, OPTIONS, CACHE,
    TRAINING_METHODS, FEATURE_VARIANTS, HORIZONS, ROLLING,
    OUT, max_mb=MAX_FEATURE_MATRIX_MB,
)
print("Завершено моделей:", int(RUN_INDEX.status.eq("complete").sum()))
print("Все статусы и причины:")
display(HTML(direct_table_html("Журнал запусков", RUN_INDEX)))


## 6. Сохранённый инференс без повторной матрицы

Проверка использует сохранённый последний пригодный момент котла и последнюю модель
каждого эксперимента. Модель не применяется к датам раньше её доступности.
Новые реальные данные требуют повторной подготовки исходной истории тем же контрактом.


In [ ]:
import joblib
index_path = OUT/"experiment_index.csv"
CURRENT_INDEX = pd.read_csv(index_path) if index_path.exists() else pd.DataFrame()
completed = CURRENT_INDEX.loc[CURRENT_INDEX.status.eq("complete")] if "status" in CURRENT_INDEX else pd.DataFrame()
latest = (completed.sort_values("test_end").groupby("experiment", sort=False).tail(1)
          if not completed.empty else pd.DataFrame())
choices_path = OUT/"validation_choices.csv"
if choices_path.exists():
    choices = pd.read_csv(choices_path)
    if not choices.empty:
        latest = choices.sort_values("validation_end").groupby(["method", "name", "horizon_hours"], sort=False).tail(1)
PREDICTIONS_NOW = []
for item in latest.itertuples():
    package = joblib.load(Path(item.folder)/"classifier.joblib")
    for unit in package["experiment"]["units"]:
        last = pd.read_parquet(Path(CACHE[unit]["folder"])/"tail.parquet")
        if last.empty or last.index[-1] < pd.Timestamp(package["available_at"]):
            continue
        result = direct_predict(package, last)
        result["unit"] = unit
        result["experiment"] = item.experiment
        PREDICTIONS_NOW.append(result.reset_index())
    del package
    gc.collect()
LATEST_PREDICTIONS = pd.concat(PREDICTIONS_NOW, ignore_index=True) if PREDICTIONS_NOW else pd.DataFrame()
display(HTML(direct_table_html("Последние доступные оценки по экспериментам", LATEST_PREDICTIONS)))


## 7. Все отчёты и сравнение

Можно запускать отдельно после прерывания: читается индекс уже сохранённых моделей,
обучение не запускается. Под стрелками модели/котла показано качество за всю тестовую
историю, а не отдельные отчёты по фолдам. Большие таблицы и ненулевая важность свёрнуты.
PNG встроены и не зависят от matplotlib backend.

В сводке Precision/Recall/F считаются по суммарным ошибкам непересекающихся test-периодов.
ROC AUC и AP показаны как среднее/min/max **отдельных** периодов: баллы разных
моделей с разными шкалами не склеиваются в один якобы общий AUC. Число доступных
версий видно в `versions`, число версий с определимым AUC/AP — в `rankable_versions`.
Средний случайный AP сравнивается на тех же определимых периодах. Для каждого горизонта
своя целевая метка и собственная доля положительных, поэтому сравнивайте также lift к baseline.

Среди baseline есть случайное угадывание, постоянный прогноз по train и **«всегда тревога»**.
Событийные показатели по отдельным периодам могут иметь неполное предаварийное окно
на границе: смотрите покрытие в отчёте. Для бюджетных вариантов дополнительно
считается единая последовательная история: состояние подтверждения/подавления
повторов не сбрасывается на границе смены модели, а порог меняется на выбранный
по прошлому validation. Эти поля имеют префикс `rolling_`.
Это диагностическая проверка; окончательная
оценка после разработки требует ещё не использованного периода.


In [ ]:
index_path = OUT/"experiment_index.csv"
if not index_path.exists():
    raise FileNotFoundError("Пока нет сохранённых запусков: выполните пункт 5.")
RUN_INDEX = pd.read_csv(index_path)
ROLLING_SUMMARY, ALL_FOLD_METRICS = render_direct_suite(
    RUN_INDEX, CONTEXTS, OUT, report_folds=REPORT_FOLDS,
)


## 8. Экспорт классификатора для дашборда

Экспорт не обучает модели: он упаковывает уже сохранённые версии, их параметры подготовки и пороги.
Копируйте на сервер целиком каталог семьи/горизонта, включая index.json и все версии.
AUTO использует выбор набора признаков по прошлой validation, а не по test.
Unit/group выбирается явно; для общей модели укажите в boilers.json имя котла из экспорта.

Новая версия формирования признаков не использует _Тип из заявок для расчёта окон.
Если веса обучались старой версией, сначала выполните обновлённые разделы 2–5:
пересчёт этого классификатора необходим, прежние результаты останутся в старом кеше.
Подготовка и пороги после экспорта фиксированы; сервер не запускает обучение.


In [ ]:
from classifier_export import export_direct_suite

EXPORT_METHODS = list(TRAINING_METHODS)  # ["unit"], ["group"] либо оба
EXPORT_VARIANT = "AUTO"      # либо "numeric", "numeric_states", "full"
EXPORT_HORIZONS = [168]      # должны быть обучены выше
EXPORT_DIR = OUT / "dashboard_export"

_exports, _export_errors = [], []
for _method in EXPORT_METHODS:
    try:
        _exports.append(export_direct_suite(OUT, output=EXPORT_DIR,
            method=_method, variant=EXPORT_VARIANT, horizons=EXPORT_HORIZONS))
    except ValueError as _error:
        _export_errors.append({"method": _method, "status": "not_exported", "reason": str(_error)})
if _export_errors:
    display(pd.DataFrame(_export_errors))
EXPORTS = pd.concat(_exports, ignore_index=True) if _exports else pd.DataFrame()
display(EXPORTS)
if EXPORTS.empty:
    raise ValueError("Нет готового экспорта. Смотрите причины выше; при старом контракте повторите обучение в обновлённом 05.")
print("На сервер копируется каталог из model_dir целиком (с версиями и index.json).")
